# AgriTech v2: multi-crop classifier and leaf/fruit/ear/pest detector

**How to run**
1. `Runtime > Change runtime type > T4 GPU`.
2. **Dry run first** (`DRY_RUN = True`, the default): `Runtime > Run all`. It takes a few images from every source and trains each model for 1 epoch (about 20-30 min, mostly the PlantVillage and PlantDoc git clones and the 1 GB wheat zip). Check the **source status table** printed after step 2 and at the end: every source should be `OK` (Roboflow ones are `SKIPPED` without a key).
3. **Full run:** set `DRY_RUN = False` and `Runtime > Run all` again. It uses a separate Drive folder, so dry-run data never mixes in.
4. Allow Google Drive access when asked. Everything (prepared data, checkpoints, models) is saved to `MyDrive/agritech_v2` (`agritech_v2_dryrun` for the dry run).
5. If Colab disconnects, run all again: finished sources are restored from Drive and training resumes from the last epoch.

A source that fails or has an unexpected layout is logged as `FAILED`, `EMPTY` or `PARTIAL` and the run continues without it.
`COMMERCIAL_SAFE = True` (default) skips non-commercial and license-unclear sources; credits for CC BY data are written to `DATASET_CREDITS.csv` and `IMAGE_ATTRIBUTIONS.csv`.

**Optional keys** (Colab sidebar > key icon > add secret, enable notebook access):
- `HF_TOKEN`: faster Hugging Face downloads, fewer rate limits.
- `ROBOFLOW_API_KEY`: adds the lettuce and olive datasets.

**Expected time on a free T4:** data prep 3-5 h (first run only), classifier 6-10 h, detector 3-5 h, spread over several sessions.

Dataset choices, licenses and rejected sources are documented in `docs/datasets.md`.

In [ ]:
# DRY_RUN = True: a few images per source + 1 epoch of each model, saved to a separate Drive folder (~20-30 min).
# Check the source status table at the end of step 2, then set DRY_RUN = False for the full run.
DRY_RUN = True
# COMMERCIAL_SAFE = True: skip non-commercial (NC) and license-unclear sources.
COMMERCIAL_SAFE = True

BACKBONE = "efficientnet_b0.ra_in1k"
IMG_SIZE = 224
STORE_MAX_SIDE = 320
MIN_IMAGES = 150
# Per-crop exceptions where the only free data is small (wheat: 122 healthy, 128 brown rust field photos)
MIN_IMAGES_BY_CROP = {"wheat": 100}
MIN_TEST = 20
CAP_PER_SOURCE_CLASS = 3000
RAW_CAP = 4000
NEAR_DUP_SIM = 0.95
LEAK_SIM = 0.97
HEAD_EPOCHS = 3
FULL_EPOCHS = 30
EPOCH_SIZE = 60000
BATCH = 96
LR_HEAD, LR_FULL = 1e-3, 2e-4
TARGET_PRECISION = 0.90
MAX_ONNX_MB = 25
YOLO_EPOCHS = 120
YOLO_PSEUDO_EPOCHS = 60
YOLO_CAP_PER_SOURCE = 2500
SEED = 42
DRY_PER_SOURCE = 0
DRY_SCAN_ROWS = 0

if DRY_RUN:
    MIN_IMAGES, MIN_TEST = 2, 1
    RAW_CAP = CAP_PER_SOURCE_CLASS = 20
    DRY_PER_SOURCE, DRY_SCAN_ROWS = 60, 3000
    HEAD_EPOCHS, FULL_EPOCHS = 1, 1
    EPOCH_SIZE = BATCH * 4
    YOLO_EPOCHS = YOLO_PSEUDO_EPOCHS = 1
    YOLO_CAP_PER_SOURCE = 20

!nvidia-smi -L
import os, sys, subprocess
from pathlib import Path

# Pin Colab's own CUDA torch so the installs below can never swap it for a CPU-only build
pin = subprocess.run([sys.executable, "-c", "import torch, torchvision; print(f'torch=={torch.__version__}'); print(f'torchvision=={torchvision.__version__}')"],
                     capture_output=True, text=True).stdout
Path("/content/torch_pin.txt").write_text(pin)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-c", "/content/torch_pin.txt",
                       "timm", "datasets", "huggingface_hub", "imagehash", "ultralytics", "onnx", "onnxruntime", "onnxslim", "onnxscript"])

# Checked in a fresh process so it sees what is on disk, not what this kernel already imported
def cuda_torch_ok():
    return subprocess.run([sys.executable, "-c", "import torch; assert torch.cuda.is_available()"], capture_output=True).returncode == 0

if not cuda_torch_ok():
    if subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
        raise RuntimeError("No GPU attached. Runtime > Change runtime type > T4 GPU, then run all cells again.")
    print("Installed torch is CPU-only, reinstalling the CUDA build...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "torch", "torchvision",
                           "--index-url", "https://download.pytorch.org/whl/cu126"])
    if not cuda_torch_ok():
        raise RuntimeError("CUDA torch still not available after reinstall. Runtime > Disconnect and delete runtime, then run all again.")
    if "torch" in sys.modules:
        print("CUDA torch installed. Restarting the runtime: run all cells again after it reconnects.")
        os.kill(os.getpid(), 9)

import torch
assert torch.cuda.is_available(), "torch cannot see the GPU. Runtime > Change runtime type > T4 GPU, then run all again."
print(f"torch {torch.__version__} | CUDA {torch.version.cuda} | {torch.cuda.get_device_name(0)} | Python {sys.version.split()[0]}")

import os, io, re, json, math, random, shutil, tarfile, zipfile, subprocess, urllib.request, zlib, time
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

random.seed(SEED)
np.random.seed(SEED)
Image.MAX_IMAGE_PIXELS = None

from google.colab import drive
drive.mount("/content/drive")
SUFFIX = "_dryrun" if DRY_RUN else ""
ROOT = Path(f"/content/drive/MyDrive/agritech_v2{SUFFIX}")
PREP_DRIVE = ROOT / "prep"
CKPT = ROOT / "checkpoints"
EXPORT = ROOT / "export"
FIGS = ROOT / "figures"
LOCAL = Path(f"/content/prep{SUFFIX}")
RAW = Path(f"/content/raw{SUFFIX}")
for d in (PREP_DRIVE, CKPT, EXPORT, FIGS, LOCAL, RAW):
    d.mkdir(parents=True, exist_ok=True)
print(f"DRY_RUN={DRY_RUN} COMMERCIAL_SAFE={COMMERCIAL_SAFE} -> saving to {ROOT}")

# One row per source: OK / PARTIAL / EMPTY / FAILED / SKIPPED, shown as a table after each download step
STATUS = {}


def set_status(name, status, images=0, classes=0, note=""):
    STATUS[name] = dict(status=status, images=images, classes=classes, note=str(note)[:300])
    if status != "OK":
        print(f"WARNING [{name}] {status}: {note}")


def status_table():
    t = pd.DataFrame.from_dict(STATUS, orient="index")
    if len(t):
        t.to_csv(ROOT / "source_status.csv")
        print(t.sort_values("status").to_string())
        bad = t[t.status.isin(["FAILED", "EMPTY", "PARTIAL"])]
        print(f"\n{len(t) - len(bad)} sources OK, {len(bad)} need attention" + (": " + ", ".join(bad.index) if len(bad) else ""))
    return t


def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


HF_TOKEN = secret("HF_TOKEN")
ROBOFLOW_API_KEY = secret("ROBOFLOW_API_KEY")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
def run(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)


if ROBOFLOW_API_KEY:
    run("pip install -q roboflow")


def download(url, dest):
    dest = Path(dest)
    if not dest.exists():
        print("Downloading", url)
        tmp = dest.with_suffix(dest.suffix + ".part")
        urllib.request.urlretrieve(url, tmp)
        tmp.rename(dest)
    return dest


IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def images_in(folder):
    return sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in IMG_EXT)


def norm(s):
    return " ".join(re.sub(r"[_\-]+", " ", str(s)).lower().split())

## 1. Source registry and taxonomy

Every class is `crop_part___condition`, e.g. `tomato_fruit___blossom_end_rot`, `rice_leaf___blast`. Non-plant images are the single class `unknown`.
Each source maps its raw labels to `(crop, part, condition)`. Labels not in a map are dropped and listed in the report, so nothing ambiguous is merged silently.

In [ ]:
HEALTHY = "healthy"


def S(name, repo, crop, part, labels, setting, license, **kw):
    return dict(name=name, repo=repo, crop=crop, part=part, labels={norm(k) if isinstance(k, str) else tuple(norm(x) for x in k): v for k, v in labels.items()},
                setting=setting, license=license, kind=kw.pop("kind", "hf"), **kw)


# Value forms: "cond" | ("part", "cond") | ("crop", "part", "cond") | None (drop)
SOURCES = [
    S("paddy_doctor", "Project-AgML/paddy_disease_classification", "rice", "leaf", {
        "bacterial_leaf_blight": "bacterial_leaf_blight", "bacterial_leaf_streak": "bacterial_leaf_streak",
        "bacterial_panicle_blight": ("panicle", "bacterial_panicle_blight"), "blast": "blast", "brown_spot": "brown_spot",
        "dead_heart": ("stem", "dead_heart"), "downy_mildew": "downy_mildew", "hispa": "hispa", "normal": HEALTHY, "tungro": "tungro",
    }, "field", "CC BY 4.0 (original IEEE DataPort)"),
    S("rice_india", "Project-AgML/rice_leaf_disease_classification_india", "rice", "leaf", {
        "Bacterialblight": "bacterial_leaf_blight", "Blast": "blast", "Brownspot": "brown_spot", "Tungro": "tungro",
    }, "mixed", "CC BY 4.0"),
    S("rice_bd", "Project-AgML/rice_disease_classification_bangladesh", "rice", "leaf", {
        "Healthy": HEALTHY, "Leaf Scald": "leaf_scald", "Rice Blast": "blast", "Rice Leaffolder": "leaffolder",
        "Rice Stripes": "stripe_virus", "Rice Tungro": "tungro",
    }, "field", "CC BY 4.0", config="raw"),
    S("rice_bact_fungal", "Project-AgML/rice_leaf_bacterial_fungal", "rice", "leaf", {
        "Bacterial Leaf Blight": "bacterial_leaf_blight", "Brown Spot": "brown_spot", "Healthy Rice Leaf": HEALTHY,
        "Leaf Blast": "blast", "Leaf scald": "leaf_scald", "Narrow Brown Leaf Spot": "narrow_brown_spot",
        "Rice Hispa": "hispa", "Sheath Blight": ("stem", "sheath_blight"),
    }, "field", "CC BY 4.0", config="raw"),
    S("dhan_shomadhan", "Project-AgML/Dhan-Shomadhan_rice_leaf_disease_classification", "rice", "leaf", {
        "Brown Spot": "brown_spot", "Browon Spot": "brown_spot", "Leaf Scaled": "leaf_scald", "Rice Blast": "blast",
        "Rice Tungro": "tungro", "Rice Turgro": "tungro", "Shath Blight": ("stem", "sheath_blight"), "Sheath Blight": ("stem", "sheath_blight"),
    }, "mixed", "CC BY 4.0"),
    S("rice_leaf_2", "Project-AgML/rice_leaf_disease_classification_2", "rice", "leaf", {
        "Bacterial leaf blight": "bacterial_leaf_blight", "Brown spot": "brown_spot", "Leaf Blast": "blast",
    }, "mixed", "CC BY 4.0"),
    S("wheat_jic", "https://zenodo.org/records/7573133/files/Long%202023%20Plant%20Path%20999%20photos.zip?download=1", "wheat", "leaf", {
        "yellow": "yellow_rust", "stripe": "yellow_rust", "brown": "brown_rust", "leaf rust": "brown_rust",
        "septoria": "septoria", "mildew": "powdery_mildew", "healthy": HEALTHY,
    }, "field", "CC BY 4.0", kind="zip_keywords"),
    S("maize_tz", "Project-AgML/maize_disease_classification", "maize", "leaf", {
        "Healthy": HEALTHY, "MLN": "lethal_necrosis", "MSV": "streak_virus",
    }, "field", "CC BY 4.0"),
    S("ccmt", "Project-AgML/crop_pest_disease_classification", None, "leaf", {
        ("cashew", "anthracnose"): "anthracnose", ("cashew", "gumosis"): ("stem", "gummosis"), ("cashew", "healthy"): HEALTHY,
        ("cashew", "leaf miner"): "leaf_miner", ("cashew", "red rust"): "red_rust",
        ("cassava", "bacterial blight"): "bacterial_blight", ("cassava", "brown spot"): "brown_leaf_spot",
        ("cassava", "green mite"): "green_mite", ("cassava", "healthy"): HEALTHY, ("cassava", "mosaic"): "mosaic_disease",
        ("maize", "fall armyworm"): "fall_armyworm", ("maize", "grasshoper"): "grasshopper_damage", ("maize", "healthy"): HEALTHY,
        ("maize", "leaf beetle"): "leaf_beetle_damage", ("maize", "streak virus"): "streak_virus",
        ("tomato", "healthy"): HEALTHY, ("tomato", "leaf curl"): "yellow_leaf_curl_virus",
        ("tomato", "septoria leaf spot"): "septoria_leaf_spot", ("tomato", "verticulium wilt"): "verticillium_wilt",
    }, "field", "CC BY 4.0", config="raw", crop_col="crop", crop_map={"Tomato": "tomato", "Cashew": "cashew", "Cassava": "cassava", "Maize": "maize"}),
    S("tom2024", "Project-AgML/TOM2024_disease_classification", None, "leaf", {
        ("tomato", "alternaria-d"): "early_blight", ("tomato", "bacterial-floundering-d"): "bacterial_wilt",
        ("tomato", "blossom-end-rot-d"): ("fruit", "blossom_end_rot"), ("tomato", "fusarium-d"): "fusarium_wilt",
        ("tomato", "healthy-fruit"): ("fruit", HEALTHY), ("tomato", "healthy-leaf"): HEALTHY,
        ("tomato", "helicoverpa-armigera-p"): ("fruit", "fruit_borer"), ("tomato", "mite-d"): "spider_mites",
        ("tomato", "sunburn-d"): ("fruit", "sunscald"), ("tomato", "tomato-late-blight-d"): "late_blight",
        ("tomato", "tuta-absoluta-p"): "leaf_miner",
        ("onion", "alternaria-d"): "purple_blotch", ("onion", "fusarium-d"): ("bulb", "fusarium_basal_rot"), ("onion", "healthy-leaf"): HEALTHY,
        ("maize", "healthy-leaf"): HEALTHY, ("maize", "curvulariosis-d"): "curvularia_leaf_spot",
        ("maize", "helminthosporiosis-d"): "northern_leaf_blight", ("maize", "rust-d"): "common_rust",
        ("maize", "spodoptera-frugiperda-p"): "fall_armyworm", ("maize", "stripe-d"): "streak_virus",
    }, "field", "CC BY 4.0", config="raw", crop_col="crop_type", crop_map={"Tomato": "tomato", "Onion": "onion", "Maize": "maize"}),
    S("corn_leaf", "Project-AgML/corn_leaf_disease_classification", "maize", "leaf", {
        "Bipolaris_Maydis": "southern_leaf_blight", "Common_Rust": "common_rust",
        "Curvularia_Lunata(wakker)boed_Spot": "curvularia_leaf_spot", "Northern-Leaf-Blight": "northern_leaf_blight",
    }, "field", "CC BY 4.0"),
    S("sugarcane", "Project-AgML/sugarcane_leaf_disease_classification", "sugarcane", "leaf", {
        "Banded Chlorosis": "banded_chlorosis", "Brown Spot": "brown_spot", "BrownRust": "brown_rust", "Grassy shoot": ("plant", "grassy_shoot"),
        "Healthy": HEALTHY, "Pokkah Boeng": "pokkah_boeng", "Sett Rot": ("stem", "sett_rot"), "Smut": ("stem", "smut"),
        "Viral Disease": "viral_disease", "Yellow Leaf": "yellow_leaf",
    }, "field", "CC BY 4.0"),
    S("cotton_bd2", "Project-AgML/cotton_leaf_disease_classification_bangladesh_2", "cotton", "leaf", {
        "Bacterial Blight": "bacterial_blight", "Curl Virus": "leaf_curl_virus", "Healthy Leaf": HEALTHY,
        "Herbicide Growth Damage": "herbicide_damage", "Leaf Hopper Jassids": "jassids", "Leaf Redding": "leaf_reddening",
        "Leaf Variegation": "variegation",
    }, "field", "CC BY 4.0", config="raw"),
    S("cotton_nc", "Project-AgML/cotton_leaf_disease_classification", "cotton", "leaf", {
        "Alternaria_Leaf": "alternaria_leaf_spot", "Bacterial_Blight": "bacterial_blight", "Fusarium_Wilt": "fusarium_wilt",
        "Healthy_Leaf": HEALTHY, "Verticillium_Wilt": "verticillium_wilt",
    }, "field", "CC BY-NC 4.0", config="raw", nc=True),
    S("soya_mh", "Project-AgML/MH_SoyaHealthVision_disease_classification_leaf", "soybean", "leaf", {
        "Caterpillar_Semilooper_Pest": "semilooper", "Frog_Leaf_Eye": "frogeye_leaf_spot", "Healthy": HEALTHY,
        "Mosaic": "mosaic_virus", "Rust": "rust", "Spectoria_Brown_Spot": "septoria_brown_spot",
    }, "field", "CC BY 4.0"),
    S("soybean_leaf", "Project-AgML/soybean_leaf_disease_classification", "soybean", "leaf", {
        "Bacterial leaf Blight": "bacterial_blight", "Healthy": HEALTHY, "Septoria_Brown_Spot": "septoria_brown_spot",
        "Vein Necrosis": "vein_necrosis",
    }, "field", "CC BY 4.0"),
    S("potato_blight", "Project-AgML/potato_leaf_blight", "potato", "leaf", {
        "earlyblt": "early_blight", "healthy": HEALTHY, "lateblt": "late_blight",
    }, "mixed", "CC BY 4.0"),
    S("potato_leaf", "Project-AgML/potato_leaf_disease_classification", "potato", "leaf", {
        "Phytopthora": "late_blight", "Healthy": HEALTHY, "Virus": "viral_disease",
    }, "field", "CC BY 4.0"),
    S("potato_tuber", "Project-AgML/potato_disease_classification", "potato", "tuber", {
        "Black Scurf": "black_scurf", "Blackspot Bruising": "bruising", "Brown Rot": "brown_rot", "Common Scab": "common_scab",
        "Dry Rot": "dry_rot", "Healthy Potatoes": HEALTHY, "Pink Rot": "pink_rot", "Soft Rot": "soft_rot",
    }, "lab", "CC BY 4.0"),
    S("tomato_quality", "Project-AgML/tomato_quality_classification", "tomato", "fruit", {
        "Fresh": HEALTHY, "Rotten": "rotten",
    }, "mixed", "CC BY 4.0", config="raw", white_bg=True),
    S("plantcity", "Project-AgML/plantcity_leaf_disease_classification", None, "leaf", {
        "apple_brown_spot": ("apple", "leaf", "brown_spot"), "apple_normal": ("apple", "leaf", HEALTHY),
        "apricot_normal": ("apricot", "leaf", HEALTHY), "apricot_shot_hole": ("apricot", "leaf", "shot_hole"),
        "bean_bean_rust_image": ("bean", "leaf", "rust"), "bean_normal_leaf": ("bean", "leaf", HEALTHY),
        "cherry_brown_spot": ("cherry", "leaf", "brown_spot"), "cherry_leaf_scorch": ("cherry", "leaf", "leaf_scorch"),
        "cherry_normal_leaf": ("cherry", "leaf", HEALTHY), "cherry_purple_leaf_spot": ("cherry", "leaf", "purple_leaf_spot"),
        "cherry_shot_hole_disease": ("cherry", "leaf", "shot_hole"),
        "corn_gray_leaf_spot": ("maize", "leaf", "gray_leaf_spot"), "corn_holcus_leaf_spot": ("maize", "leaf", "holcus_leaf_spot"),
        "corn_normal_leaf": ("maize", "leaf", HEALTHY),
        "fig_normal_leaf": ("fig", "leaf", HEALTHY), "fig_rust_leaf": ("fig", "leaf", "rust"),
        "grape_anthracnose_leaf": ("grape", "leaf", "anthracnose"), "grape_downy_mildew_leaf": ("grape", "leaf", "downy_mildew"),
        "grape_mites_leaf_disease": ("grape", "leaf", "erineum_mite"), "grape_normal_leaf": ("grape", "leaf", HEALTHY),
        "grape_powdery_mildew_leaf": ("grape", "leaf", "powdery_mildew"),
        "lokat_leaf_spot": ("loquat", "leaf", "leaf_spot"), "lokat_normal_leaf": ("loquat", "leaf", HEALTHY),
        "pear_black_spot_leaf_disease": ("pear", "leaf", "black_spot"), "pear_fire_blight": ("pear", "leaf", "fire_blight"),
        "pear_normal_leaf": ("pear", "leaf", HEALTHY), "persimmons_brown_spot": ("persimmon", "leaf", "brown_spot"),
        "tomato_bacterial_spot": ("tomato", "leaf", "bacterial_spot"), "tomato_early_blight": ("tomato", "leaf", "early_blight"),
        "tomato_fusarium_wilt": ("tomato", "leaf", "fusarium_wilt"), "tomato_healthy_leaf": ("tomato", "leaf", HEALTHY),
        "tomato_late_blight": ("tomato", "leaf", "late_blight"), "tomato_leaf_curl": ("tomato", "leaf", "yellow_leaf_curl_virus"),
        "tomato_leaf_miner": ("tomato", "leaf", "leaf_miner"), "tomato_leaf_mold": ("tomato", "leaf", "leaf_mold"),
        "tomato_septoria_leaf": ("tomato", "leaf", "septoria_leaf_spot"), "tomato_spider_mites": ("tomato", "leaf", "spider_mites"),
        "tomato_verticillium_wilt": ("tomato", "leaf", "verticillium_wilt"),
        "walnut_anthracnose_leaf_disease": ("walnut", "leaf", "anthracnose"), "walnut_leaf_gall_mite": ("walnut", "leaf", "erineum_mite"),
        "walnut_normal_leaf": ("walnut", "leaf", HEALTHY),
    }, "field", "CC BY 4.0", config="raw"),
    S("chili_cold", "Project-AgML/COLD_chili_leaf_disease_classification", "pepper", "leaf", {
        "cercospora": "cercospora_leaf_spot", "healthy": HEALTHY, "mites_and_trips": "mites_thrips",
        "nutritional": "nutrient_deficiency", "powdery mildew": "powdery_mildew",
    }, "field", "CC BY 4.0", config="raw"),
    S("chili_virus", "Project-AgML/chili_pepper_virus_classification", "pepper", "leaf", {
        "healthy": HEALTHY, "bbwv2_infected": "broad_bean_wilt_virus",
    }, "mixed", "CC BY 4.0"),
    S("vegnet_quality", "Project-AgML/VegNet_quality_classification", None, "fruit", {
        "Damaged": "damaged", "Ripe": HEALTHY, "Unripe": HEALTHY,
    }, "lab", "CC BY 4.0", crop_col="crop_type",
       crop_map={"Bell Pepper": "pepper", "Chile Pepper": "pepper", "New Mexico Green Chile": "pepper", "Tomato": "tomato"}),
    S("eggplant", "Project-AgML/eggplant_disease_classification", "eggplant", "leaf", {
        "Healthy Leaf": HEALTHY, "Insect Pest Disease": "insect_damage", "Leaf Spot Disease": "leaf_spot",
        "Mosaic Virus Disease": "mosaic_virus", "White Mold Disease": "white_mold", "Wilt Disease": "wilt",
    }, "field", "CC BY 4.0", config="raw"),
    S("eggplant_nc", "Project-AgML/eggplant_leaf_disease_classification", "eggplant", "leaf", {
        "Aphids": "aphids", "Cercospora Leaf Spot": "cercospora_leaf_spot", "Defect Eggplant": ("fruit", "damaged"),
        "Flea Beetles": "flea_beetle", "Fresh Eggplant": ("fruit", HEALTHY), "Fresh Eggplant Leaf": HEALTHY,
        "Leaf Wilt": "wilt", "Phytophthora Blight": "phytophthora_blight", "Powdery Mildew": "powdery_mildew",
        "Tobacco Mosaic Virus": "mosaic_virus",
    }, "field", "CC BY-NC 4.0", config="raw", nc=True),
    S("brinjal_fruit", "Project-AgML/BrinjalFruitX_disease_classification", "eggplant", "fruit", {
        "Brinjal Fruit Creaking": "fruit_cracking", "Healty Brinjal": HEALTHY, "Phomopsis Bright": "phomopsis_blight",
        "Shoot and Fruit Borer": "fruit_borer", "Wet Rot": "wet_rot",
    }, "field", "CC BY 4.0"),
    S("veg_leaf_bd", "Project-AgML/plant_leaf_disease_classification", None, "leaf", {
        "Anthracnose": "anthracnose", "Anthracnose lesions": "anthracnose", "Black Rot": "black_rot",
        "Downey mildew": "downy_mildew", "Downy mildew": "downy_mildew", "Eggplant Cercopora leaf spot": "cercospora_leaf_spot",
        "Eggplant begomovirus": "mosaic_virus", "Eggplant fresh leaf": HEALTHY, "Eggplant verticillium wilt": "verticillium_wilt",
        "Fresh leaf": HEALTHY, "Fusarium wilt": "fusarium_wilt", "Mosaic virus": "mosaic_virus",
        "Tomato Bacterial spot": "bacterial_spot", "Tomato Fresh leaf": HEALTHY, "Tomato leaf curl virus": "yellow_leaf_curl_virus",
        "Tomato spotted wilt": "spotted_wilt_virus",
    }, "field", "CC BY 4.0", crop_col="plant_type",
       crop_map={"Bitter Gourd": "bitter_gourd", "Bottle gourd": "bottle_gourd", "Cauliflower": "cauliflower", "Cucumber": "cucumber", "Eggplant": "eggplant", "Tomato": "tomato"}),
    S("watermelon", "Project-AgML/watermelon_disease_classification", "watermelon", "leaf", {
        "Anthracnose": "anthracnose", "Downy_Mildew": "downy_mildew", "Healthy": HEALTHY, "Mosaic_Virus": "mosaic_virus",
    }, "field", "CC BY 4.0", config="raw"),
    S("vegnet_cauliflower", "Project-AgML/VegNet_cauliflower_disease_classification", "cauliflower", "head", {
        "Bacterial spot rot": "bacterial_spot_rot", "Black Rot": "black_rot", "Downy Mildew": "downy_mildew", "No disease": HEALTHY,
    }, "field", "CC BY 4.0", config="raw"),
    S("cauliflower_leaf", "Project-AgML/cauliflower_leaf_disease_classification", "cauliflower", "leaf", {
        "Black Rot": "black_rot", "Healthy": HEALTHY, "Insect Hole": "insect_damage",
    }, "field", "CC BY 4.0"),
    S("onion_cold", "Project-AgML/COLD_onion_leaf_disease_classification", "onion", "leaf", {
        "Iris yellow virus": "iris_yellow_spot_virus", "Stemphylium leaf blight and collectrichum leaf blight": "stemphylium_blight",
        "healthy": HEALTHY, "purple blotch": "purple_blotch",
    }, "field", "CC BY 4.0", config="raw"),
    S("ibeans", "Project-AgML/bean_disease_uganda", "bean", "leaf", {
        "angular_leaf_spot": "angular_leaf_spot", "bean_rust": "rust", "healthy": HEALTHY,
    }, "field", "CC BY 4.0"),
    S("bean_tz", "Project-AgML/bean_disease_classification_tanzania", "bean", "leaf", {
        "anthracnose": "anthracnose", "healthy": HEALTHY, "rust": "rust",
    }, "field", "CC BY 4.0"),
    S("bean_cowpea", "Project-AgML/bean_cowpea_leaf_disease_classification", None, "leaf", {
        "Bacterial wilt": "bacterial_wilt", "Fresh Leaf": HEALTHY, "Mosaic Virus": "mosaic_virus", "Rust": "rust",
        "Septoria leaf spot": "septoria_leaf_spot",
    }, "field", "CC BY 4.0", crop_col="crop_type", crop_map={"bean": "bean", "cowpea": "cowpea"}),
    S("appleleaf9", "Project-AgML/appleleaf9_disease_classification", "apple", "leaf", {
        "Alternaria leaf spot": "alternaria_leaf_spot", "Brown spot": "brown_spot", "Frogeye leaf spot": "frogeye_leaf_spot",
        "Grey spot": "grey_spot", "Health": HEALTHY, "Mosaic": "mosaic", "Powdery mildew": "powdery_mildew", "Rust": "rust", "Scab": "scab",
    }, "field", "CC BY 4.0"),
    S("apple_leaf", "Project-AgML/apple_leaf_disease_classification", "apple", "leaf", {
        "Alternaria": "alternaria_leaf_spot", "Apple_Mosaic": "mosaic", "Healthy": HEALTHY,
    }, "field", "CC BY 4.0"),
    S("plant_pathology_2021", "timm/plant-pathology-2021", "apple", "leaf", {
        "healthy": HEALTHY, "scab": "scab", "frog_eye_leaf_spot": "frogeye_leaf_spot", "rust": "rust", "powdery_mildew": "powdery_mildew",
    }, "field", "Apache-2.0 per mirror; original is Kaggle competition data (unclear)", label_col="label_names", commercial=False),
    S("gvlid", "Project-AgML/gvlid_grapevine_leaf_disease", "grape", "leaf", {
        "black_rot": "black_rot", "esca": "esca", "healthy": HEALTHY, "leaf_blight": "leaf_blight",
    }, "field", "CC BY 4.0"),
    S("grape_leaf", "Project-AgML/grape_leaf_disease_classification", "grape", "leaf", {
        "Bacterial Leaf Spot": "bacterial_leaf_spot", "Downy Mildew": "downy_mildew", "Healthy Leaves": HEALTHY, "Powdery Mildew": "powdery_mildew",
    }, "mixed", "CC BY 4.0"),
    S("lemon_leaf", "Project-AgML/lemon_leaf_disease_classification", "citrus", "leaf", {
        "Anthracnose": "anthracnose", "Bacterial Blight": "bacterial_blight", "Citrus Canker": "canker", "Curl Virus": "leaf_curl_virus",
        "Deficiency Leaf": "nutrient_deficiency", "Healthy Leaf": HEALTHY, "Sooty Mould": "sooty_mould", "Spider Mites": "spider_mites",
    }, "field", "CC BY 4.0", config="raw"),
    S("citrus_fruit_leaf", "Project-AgML/citrus_fruit_leaf_disease_classification", "citrus", None, {
        "black_spot": "black_spot", "canker": "canker", "greening": "greening", "healthy": HEALTHY, "melanose": "melanose", "scab": "scab",
    }, "field", "CC BY 4.0", part_col="plant_part"),
    S("banana_tz", "Project-AgML/banana_disease_classification_tanzania", "banana", "leaf", {
        "black_sigatoka": "sigatoka", "fusarium_wilt": "fusarium_wilt", "healthy": HEALTHY,
    }, "field", "CC0"),
    S("banana_lsd", "Project-AgML/BananaLSD_leaf_disease_classification", "banana", "leaf", {
        "cordana": "cordana_leaf_spot", "healthy": HEALTHY, "pestalotiopsis": "pestalotiopsis_leaf_spot", "sigatoka": "sigatoka",
    }, "field", "CC BY 4.0", config="raw"),
    S("banana_leaf", "Project-AgML/banana_leaf_disease_classification", "banana", "leaf", {
        "healthy": HEALTHY, "segatoka": "sigatoka", "xamthomonas": "xanthomonas_wilt",
    }, "field", "CC BY 4.0"),
    S("mango_leafbd", "Project-AgML/MangoLeafBD_disease_classification", "mango", "leaf", {
        "Anthracnose": "anthracnose", "Bacterial Canker": "bacterial_canker", "Cutting Weevil": "cutting_weevil", "Die Back": "die_back",
        "Gall Midge": "gall_midge", "Healthy": HEALTHY, "Powdery Mildew": "powdery_mildew", "Sooty Mould": "sooty_mould",
    }, "field", "CC BY-NC 3.0", nc=True),
    S("coffee_arabica", "Project-AgML/arabica_coffee_leaf_disease_classification", "coffee", "leaf", {
        "Cerscospora": "cercospora_leaf_spot", "Healthy": HEALTHY, "Leaf_rust": "leaf_rust", "Miner": "leaf_miner", "Phoma": "phoma_leaf_spot",
    }, "field", "CC BY 4.0"),
    S("tea_leafbd", "Project-AgML/teaLeafBD_disease_classification", "tea", "leaf", {
        "Brown Blight": "brown_blight", "Gray Blight": "gray_blight", "Green mirid bug": "green_mirid_bug", "Healthy leaf": HEALTHY,
        "Helopeltis": "helopeltis", "Red spider": "red_spider_mite", "Tea algal leaf spot": "algal_leaf_spot",
    }, "field", "CC BY 4.0"),
    S("tea_bd2", "Project-AgML/tea_leaf_disease_classification_bangladesh_2", "tea", "leaf", {
        "Healthy_Leaf": HEALTHY, "Helopeltis": "helopeltis", "Red_Rust": "red_rust",
    }, "field", "CC BY 4.0"),
    S("groundnut", "Project-AgML/groundnut_leaf_disease_classification", "groundnut", "leaf", {
        "ALTERNARIA LEAF SPOT": "alternaria_leaf_spot", "HEALTHY": HEALTHY, "LEAF SPOT": "leaf_spot", "ROSETTE": "rosette_virus", "RUST": "rust",
    }, "field", "CC BY 4.0"),
    S("guava", "Project-AgML/guava_disease_classification", "guava", None, {
        "Anthracnose": "anthracnose", "Canker": "canker", "Dot": "dot", "Healthy": HEALTHY, "Rust": "rust", "Scab": "scab",
        "Styler end root": "styler_end_rot",
    }, "field", "CC BY 4.0", config="raw", part_col="plant_part"),
    S("papaya", "Project-AgML/papaya_leaf_disease_classification_bangladesh", "papaya", "leaf", {
        "Healthy Leaf": HEALTHY, "Leaf Curl": "leaf_curl_virus", "Mealybug": "mealybug", "Mite Disease": "mites",
        "Mosaic": "mosaic_virus", "Ring Spot": "ringspot_virus",
    }, "field", "CC BY 4.0", config="raw"),
    S("pomegranate", "Project-AgML/pomegranate_disease_classification_india", "pomegranate", "fruit", {
        "Alternaria": "alternaria_fruit_rot", "Anthracnose": "anthracnose", "Bacterial_Blight": "bacterial_blight",
        "Cercospora": "cercospora_fruit_spot", "Healthy": HEALTHY,
    }, "field", "CC BY 4.0"),
    S("okra", "Project-AgML/OkraDiseaseNet_disease_classification", "okra", "leaf", {
        "Alternaria Leaf Spot": "alternaria_leaf_spot", "Cercospora Leaf Spot": "cercospora_leaf_spot", "Downy Mildew": "downy_mildew",
        "Healthy": HEALTHY, "Leaf curly virus": "leaf_curl_virus", "Phyllosticta leaf spot": "phyllosticta_leaf_spot",
    }, "field", "CC BY 4.0"),
    S("sunflower", "Project-AgML/sunflower_disease_classification", "sunflower", "leaf", {
        "Downy_mildew": "downy_mildew", "Fresh_leaf": HEALTHY, "Gray_mold": "gray_mold", "Leaf_scars": "leaf_scars",
    }, "field", "CC BY 4.0"),
    S("lentil", "Project-AgML/lentil_disease_classification", "lentil", "leaf", {
        "Ascochyta blight": "ascochyta_blight", "Lentil Rust": "rust", "Normal": HEALTHY, "Powdery Mildew": "powdery_mildew",
    }, "field", "CC BY 4.0"),
    S("fresh_rotten", "Project-AgML/fresh_rotten_fruit_classification", None, "fruit", {
        "fresh": HEALTHY, "rotten": "rotten",
    }, "lab", "CC BY 4.0", config="raw", crop_col="fruit_type", white_bg=True,
       crop_map={"jujube": "jujube", "guava": "guava", "grape": "grape", "apple": "apple", "pomegranate": "pomegranate",
                 "banana": "banana", "orange": "citrus", "strawberry": "strawberry"}),
    S("fruitnet", "Project-AgML/FruitNet_quality_classification", None, "fruit", {
        "Good": HEALTHY, "Bad": "rotten",
    }, "mixed", "CC BY 4.0", crop_col="crop",
       crop_map={"Apple": "apple", "Banana": "banana", "Guava": "guava", "Lemon": "citrus", "Lime": "citrus", "Orange": "citrus", "Pomegranate": "pomegranate"}),
    S("fruits360", "PedroSampaio/fruits-360", None, "fruit", {}, "lab", "MIT", kind="fruits360", white_bg=True, cap=800),
    S("plantvillage", "https://github.com/spMohanty/PlantVillage-Dataset.git", None, "leaf", {}, "lab", "CC BY 4.0 (redistributed)", kind="plantvillage"),
    S("plantdoc", "https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset.git", None, "leaf", {}, "field", "CC BY 4.0", kind="plantdoc"),
    S("roboflow_lettuce", "yssp/disease-lettuce-2", "lettuce", "leaf", {
        "Bacterial": "bacterial_leaf_spot", "Downy_mildew": "downy_mildew", "Powdery_mildew": "powdery_mildew",
        "Septoria_Blight": "septoria_leaf_spot", "healthy": HEALTHY, "lettuce mosaic virus": "mosaic_virus",
        "Lettuce - Anthracnose": "anthracnose",
    }, "field", "CC BY 4.0", kind="roboflow"),
    S("roboflow_olive", "baraah/olive-s-leaf-diseases-tvce5", "olive", "leaf", {
        "healthy": HEALTHY, "Aculus olearius": "aculus_olearius", "olive peacock spot": "peacock_spot", "peacock spot": "peacock_spot",
        "knot disease": None,
    }, "field", "CC BY 4.0", kind="roboflow"),
]

# Negatives for the "unknown" class. All are CC0 or CC BY; Open Images is filtered per image to CC BY 2.0
OI_IMAGES_CSV = "https://storage.googleapis.com/openimages/2018_04/validation/validation-images-with-rotation.csv"
OI_LABELS_CSV = "https://storage.googleapis.com/openimages/v7/oidv7-val-annotations-human-imagelabels.csv"
OI_CLASSES_CSV = "https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions.csv"
OI_IMAGE_URL = "https://open-images-dataset.s3.amazonaws.com/validation/{}.jpg"
CROP_WORDS = (r"fruit|vegetable|food|produce|dish|salad|crop|farm|field|agriculture|agricultural|orchard|vineyard|harvest|grain|seed|"
              r"apple|banana|berry|berries|grape|citrus|orange|lemon|lime|mango|pear|peach|cherry|plum|tomato|potato|pepper|"
              r"cucumber|squash|pumpkin|melon|eggplant|cabbage|cauliflower|broccoli|lettuce|onion|carrot|bean|pea|corn|maize|"
              r"wheat|rice|barley|cotton|coffee|tea|cassava|sugarcane|soybean|olive|guava|papaya|pomegranate|fig|walnut|date|leaf|leaves")
FLORA_WORDS = r"flower|tree|houseplant|shrub|grass|lawn|forest|moss|fern|bush|hedge|woody plant|flowering plant|bouquet"

NEGATIVES = [
    dict(name="neg_openimages_objects", kind="openimages", mode="objects", cap=6000, license="CC BY 2.0 (per image, Open Images)"),
    dict(name="neg_openimages_flora", kind="openimages", mode="flora", cap=3000, license="CC BY 2.0 (per image, Open Images)"),
    dict(name="neg_polyhaven_textures", kind="polyhaven", crops_per_texture=6, cap=2400, license="CC0 (Poly Haven)"),
    dict(name="neg_deepweeds", kind="hf", repo="Project-AgML/DeepWeeds_classification", cap=6000, license="CC BY 4.0"),
    dict(name="neg_pv_background", kind="hf", repo="Project-AgML/Plant_leaf_diseases_dataset_classification", config="raw", only_label="Background_without_leaves", cap=2000, license="CC BY 4.0"),
    dict(name="neg_trees", kind="hf", repo="Project-AgML/tree_species_classification_slovak_normal_cropped", cap=2000, license="CC BY 4.0"),
    dict(name="neg_hands", kind="hf_yolo_images", repo="LibreYOLO/hand-gestures-jps7z", cap=2000, license="CC BY 4.0"),
]

PV_MAP = {
    "Apple___Apple_scab": ("apple", "leaf", "scab"), "Apple___Black_rot": ("apple", "leaf", "black_rot"),
    "Apple___Cedar_apple_rust": ("apple", "leaf", "rust"), "Apple___healthy": ("apple", "leaf", HEALTHY),
    "Blueberry___healthy": ("blueberry", "leaf", HEALTHY),
    "Cherry_(including_sour)___Powdery_mildew": ("cherry", "leaf", "powdery_mildew"), "Cherry_(including_sour)___healthy": ("cherry", "leaf", HEALTHY),
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": ("maize", "leaf", "gray_leaf_spot"), "Corn_(maize)___Common_rust_": ("maize", "leaf", "common_rust"),
    "Corn_(maize)___Northern_Leaf_Blight": ("maize", "leaf", "northern_leaf_blight"), "Corn_(maize)___healthy": ("maize", "leaf", HEALTHY),
    "Grape___Black_rot": ("grape", "leaf", "black_rot"), "Grape___Esca_(Black_Measles)": ("grape", "leaf", "esca"),
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)": ("grape", "leaf", "leaf_blight"), "Grape___healthy": ("grape", "leaf", HEALTHY),
    "Orange___Haunglongbing_(Citrus_greening)": ("citrus", "leaf", "greening"),
    "Peach___Bacterial_spot": ("peach", "leaf", "bacterial_spot"), "Peach___healthy": ("peach", "leaf", HEALTHY),
    "Pepper,_bell___Bacterial_spot": ("pepper", "leaf", "bacterial_spot"), "Pepper,_bell___healthy": ("pepper", "leaf", HEALTHY),
    "Potato___Early_blight": ("potato", "leaf", "early_blight"), "Potato___Late_blight": ("potato", "leaf", "late_blight"),
    "Potato___healthy": ("potato", "leaf", HEALTHY), "Raspberry___healthy": ("raspberry", "leaf", HEALTHY),
    "Soybean___healthy": ("soybean", "leaf", HEALTHY), "Squash___Powdery_mildew": ("squash", "leaf", "powdery_mildew"),
    "Strawberry___Leaf_scorch": ("strawberry", "leaf", "leaf_scorch"), "Strawberry___healthy": ("strawberry", "leaf", HEALTHY),
    "Tomato___Bacterial_spot": ("tomato", "leaf", "bacterial_spot"), "Tomato___Early_blight": ("tomato", "leaf", "early_blight"),
    "Tomato___Late_blight": ("tomato", "leaf", "late_blight"), "Tomato___Leaf_Mold": ("tomato", "leaf", "leaf_mold"),
    "Tomato___Septoria_leaf_spot": ("tomato", "leaf", "septoria_leaf_spot"),
    "Tomato___Spider_mites Two-spotted_spider_mite": ("tomato", "leaf", "spider_mites"), "Tomato___Target_Spot": ("tomato", "leaf", "target_spot"),
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": ("tomato", "leaf", "yellow_leaf_curl_virus"),
    "Tomato___Tomato_mosaic_virus": ("tomato", "leaf", "mosaic_virus"), "Tomato___healthy": ("tomato", "leaf", HEALTHY),
}

PLANTDOC_MAP = {norm(k): v for k, v in {
    "Apple Scab Leaf": ("apple", "leaf", "scab"), "Apple leaf": ("apple", "leaf", HEALTHY), "Apple rust leaf": ("apple", "leaf", "rust"),
    "Bell_pepper leaf": ("pepper", "leaf", HEALTHY), "Bell_pepper leaf spot": ("pepper", "leaf", "bacterial_spot"),
    "Blueberry leaf": ("blueberry", "leaf", HEALTHY), "Cherry leaf": ("cherry", "leaf", HEALTHY),
    "Corn Gray leaf spot": ("maize", "leaf", "gray_leaf_spot"), "Corn leaf blight": ("maize", "leaf", "northern_leaf_blight"),
    "Corn rust leaf": ("maize", "leaf", "common_rust"), "Peach leaf": ("peach", "leaf", HEALTHY), "Potato leaf": ("potato", "leaf", HEALTHY),
    "Potato leaf early blight": ("potato", "leaf", "early_blight"), "Potato leaf late blight": ("potato", "leaf", "late_blight"),
    "Raspberry leaf": ("raspberry", "leaf", HEALTHY), "Soyabean leaf": ("soybean", "leaf", HEALTHY), "Soybean leaf": ("soybean", "leaf", HEALTHY),
    "Squash Powdery mildew leaf": ("squash", "leaf", "powdery_mildew"), "Strawberry leaf": ("strawberry", "leaf", HEALTHY),
    "Tomato Early blight leaf": ("tomato", "leaf", "early_blight"), "Tomato Septoria leaf spot": ("tomato", "leaf", "septoria_leaf_spot"),
    "Tomato leaf": ("tomato", "leaf", HEALTHY), "Tomato leaf bacterial spot": ("tomato", "leaf", "bacterial_spot"),
    "Tomato leaf late blight": ("tomato", "leaf", "late_blight"), "Tomato leaf mosaic virus": ("tomato", "leaf", "mosaic_virus"),
    "Tomato leaf yellow virus": ("tomato", "leaf", "yellow_leaf_curl_virus"), "Tomato mold leaf": ("tomato", "leaf", "leaf_mold"),
    "Tomato two spotted spider mites leaf": ("tomato", "leaf", "spider_mites"),
    "grape leaf": ("grape", "leaf", HEALTHY), "grape leaf black rot": ("grape", "leaf", "black_rot"),
}.items()}

# Fruits-360 variety -> (crop, part); anything not listed is skipped
FRUITS360_PREFIX = [
    ("apple", "apple", "fruit"), ("apricot", "apricot", "fruit"), ("banana", "banana", "fruit"), ("blueberry", "blueberry", "fruit"),
    ("cauliflower", "cauliflower", "head"), ("cherry", "cherry", "fruit"), ("corn husk", None, None), ("corn", "maize", "ear"),
    ("cucumber", "cucumber", "fruit"), ("dates", "date_palm", "fruit"), ("eggplant", "eggplant", "fruit"), ("fig", "fig", "fruit"),
    ("grapefruit", "citrus", "fruit"), ("grape", "grape", "fruit"), ("guava", "guava", "fruit"), ("lemon", "citrus", "fruit"),
    ("limes", "citrus", "fruit"), ("clementine", "citrus", "fruit"), ("mandarine", "citrus", "fruit"), ("orange", "citrus", "fruit"),
    ("tangelo", "citrus", "fruit"), ("pomelo", "citrus", "fruit"), ("kumquats", "citrus", "fruit"), ("mango", "mango", "fruit"),
    ("onion", "onion", "bulb"), ("papaya", "papaya", "fruit"), ("peach", "peach", "fruit"), ("nectarine", "peach", "fruit"),
    ("pear", "pear", "fruit"), ("pepper", "pepper", "fruit"), ("plum", "plum", "fruit"), ("pomegranate", "pomegranate", "fruit"),
    ("potato sweet", None, None), ("potato", "potato", "tuber"), ("raspberry", "raspberry", "fruit"), ("strawberry wedge", None, None),
    ("strawberry", "strawberry", "fruit"), ("tomato", "tomato", "fruit"), ("watermelon", "watermelon", "fruit"), ("kaki", "persimmon", "fruit"),
    ("quince", "quince", "fruit"), ("walnut", "walnut", "nut"),
]


def resolve(spec, raw_label, crop_raw=None, part_raw=None):
    key = norm(raw_label)
    crop = spec["crop"]
    if spec.get("crop_col"):
        crop = spec["crop_map"].get(str(crop_raw).strip())
        if crop is None:
            return None
    val = spec["labels"].get((norm(crop), key), spec["labels"].get(key)) if crop else spec["labels"].get(key)
    if val is None:
        return None
    part = spec["part"] or (norm(part_raw) if part_raw is not None else "leaf")
    if isinstance(val, str):
        return crop, part, val
    if len(val) == 2:
        return crop, val[0], val[1]
    return val


def class_name(crop, part, cond):
    return "unknown" if crop == "unknown" else f"{crop}_{part}___{cond}"


for s in SOURCES:
    s.setdefault("commercial", not s.get("nc"))
if COMMERCIAL_SAFE:
    excluded = [s["name"] for s in SOURCES if not s["commercial"]]
    SOURCES = [s for s in SOURCES if s["commercial"]]
    print("COMMERCIAL_SAFE: excluded", excluded)
SPEC = {s["name"]: s for s in SOURCES}
print(len(SOURCES), "classification sources,", len(NEGATIVES), "negative sources")

## 2. Download and prepare images (resumable)

Each source is streamed, mapped, resized to at most 320 px and saved as JPEG. Finished sources are packed to Drive, so a reconnect only restores them.
Pre-augmented "augmented" configs are never used.

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def save_img(img, out):
    img = ImageOps.exif_transpose(img).convert("RGB")
    img.thumbnail((STORE_MAX_SIDE, STORE_MAX_SIDE))
    out.parent.mkdir(parents=True, exist_ok=True)
    img.save(out, quality=90)


def row(out, name, crop, part, cond, cls, setting, group_hint="", seg="", white_bg=False, official_split="", attribution=""):
    return dict(path=str(out), source=name, crop=crop, part=part, cond=cond, cls=cls, setting=setting, group_hint=group_hint,
                seg=str(seg), white_bg=white_bg, official_split=official_split, attribution=attribution)


def neg_row(out, name, hint="", attribution=""):
    return row(out, name, "unknown", "none", "unknown", "unknown", "neg", group_hint=hint, attribution=attribution)


def enough(rows, scanned=0):
    # Dry run: stop early once a source has proven it downloads and maps correctly
    return DRY_RUN and (len(rows) >= DRY_PER_SOURCE or scanned >= DRY_SCAN_ROWS)


def finish(name, rows, unmapped, problem=""):
    top_unmapped = dict(Counter(unmapped).most_common(8))
    if not rows:
        set_status(name, "EMPTY", note=problem or f"no image matched the label map; most common unmapped labels: {top_unmapped}")
        return
    frame = pd.DataFrame(rows)
    (LOCAL / name).mkdir(parents=True, exist_ok=True)
    frame.to_csv(LOCAL / name / "manifest.csv", index=False)
    with open(LOCAL / name / "unmapped.json", "w") as f:
        json.dump({str(k): v for k, v in unmapped.items()}, f)
    if problem:
        set_status(name, "PARTIAL", len(frame), frame.cls.nunique(), f"{problem} (kept for this session only, retried next session)")
        return
    tmp = PREP_DRIVE / f"{name}.tar.part"
    with tarfile.open(tmp, "w") as t:
        t.add(LOCAL / name, arcname=name)
    tmp.rename(PREP_DRIVE / f"{name}.tar")
    set_status(name, "OK", len(frame), frame.cls.nunique(), f"unmapped labels: {top_unmapped}" if unmapped else "")


def restore(name):
    manifest = LOCAL / name / "manifest.csv"
    tar = PREP_DRIVE / f"{name}.tar"
    if not manifest.exists() and tar.exists():
        try:
            with tarfile.open(tar) as t:
                t.extractall(LOCAL, filter="data")
        except Exception as e:
            print(f"WARNING [{name}] cached archive is unreadable ({e!r}); downloading again")
            tar.unlink(missing_ok=True)
            shutil.rmtree(LOCAL / name, ignore_errors=True)
            return False
    if manifest.exists():
        frame = pd.read_csv(manifest, keep_default_na=False)
        set_status(name, "OK", len(frame), frame.cls.nunique(), "restored from Drive")
        return True
    return False


def undecoded(ds):
    # Keep images as raw bytes so rows that get skipped are never decoded
    try:
        return ds.decode(False)
    except AttributeError:
        from datasets import Image as HFImage
        return ds.cast_column("image", HFImage(decode=False))


def hf_rows(repo, config=None, errors=None):
    from datasets import load_dataset, get_dataset_split_names
    for split in get_dataset_split_names(repo, config):
        try:
            ds = load_dataset(repo, config, split=split, streaming=True)
            names = {k: getattr(v, "names", None) for k, v in ds.features.items()}
            ds = undecoded(ds)
            if DRY_RUN:
                ds = ds.shuffle(seed=SEED, buffer_size=300)
            for r in ds:
                for k, n in names.items():
                    if n and isinstance(r.get(k), int):
                        r[k] = n[r[k]]
                    elif n and isinstance(r.get(k), list):
                        r[k] = [n[i] if isinstance(i, int) else i for i in r[k]]
                yield split, r
        except Exception as e:
            print(f"WARNING {repo} split '{split}' stopped: {e!r}")
            if errors is not None:
                errors.append(f"split {split}: {e!r}"[:200])


def open_hf_image(v):
    if isinstance(v, dict):
        if v.get("bytes"):
            return Image.open(io.BytesIO(v["bytes"]))
        return Image.open(v["path"])
    return v


def ingest_hf(spec):
    name, rows, counts, unmapped, errors = spec["name"], [], Counter(), Counter(), []
    cap = min(spec.get("cap", RAW_CAP), RAW_CAP)
    label_col = spec.get("label_col", "label")
    for i, (split, r) in enumerate(hf_rows(spec["repo"], spec.get("config"), errors)):
        if enough(rows, i):
            break
        raw = r.get(label_col)
        if isinstance(raw, list):
            if len(raw) != 1:
                unmapped[("multi", str(raw))] += 1
                continue
            raw = raw[0]
        crop_raw = r.get(spec["crop_col"]) if spec.get("crop_col") else None
        part_raw = r.get(spec["part_col"]) if spec.get("part_col") else None
        res = resolve(spec, raw, crop_raw, part_raw)
        if res is None:
            unmapped[(str(crop_raw), str(raw))] += 1
            continue
        cls = class_name(*res)
        if counts[cls] >= cap:
            continue
        out = LOCAL / name / cls / f"{i:07d}.jpg"
        try:
            save_img(open_hf_image(r["image"]), out)
        except Exception:
            continue
        counts[cls] += 1
        rows.append(row(out, name, *res, cls, spec["setting"], white_bg=spec.get("white_bg", False)))
    finish(name, rows, unmapped, "; ".join(errors))


def ingest_zip_keywords(spec):
    name = spec["name"]
    z = download(spec["repo"], RAW / f"{name}.zip")
    dest = RAW / name
    if not dest.exists():
        with zipfile.ZipFile(z) as f:
            f.extractall(dest)
    files = images_in(dest)
    folders = sorted({str(p.parent.relative_to(dest)) for p in files})
    print(f"[{name}] {len(files)} images in folders: {folders[:20]}")
    rows, unmapped = [], Counter()
    for i, p in enumerate(files):
        if enough(rows):
            break
        text = norm(str(p.relative_to(dest)))
        cond = next((v for k, v in spec["labels"].items() if k in text), None)
        if cond is None:
            unmapped[str(p.parent.name)] += 1
            continue
        cls = class_name(spec["crop"], spec["part"], cond)
        out = LOCAL / name / cls / f"{i:06d}.jpg"
        try:
            save_img(Image.open(p), out)
        except Exception:
            continue
        rows.append(row(out, name, spec["crop"], spec["part"], cond, cls, spec["setting"]))
    finish(name, rows, unmapped, "" if rows else f"no folder name matched the keywords {list(spec['labels'])}; folders: {folders[:10]}")


def ingest_fruits360(spec):
    name, rows, counts, unmapped, errors = spec["name"], [], Counter(), Counter(), []
    for i, (split, r) in enumerate(hf_rows(spec["repo"], errors=errors)):
        if enough(rows, i):
            break
        variety = norm(r["label"])
        hit = next(((c, p) for pre, c, p in FRUITS360_PREFIX if variety.startswith(pre)), None)
        if hit is None or hit[0] is None:
            unmapped[variety] += 1
            continue
        crop, part = hit
        cls = class_name(crop, part, HEALTHY)
        if counts[cls] >= min(spec["cap"], RAW_CAP):
            continue
        out = LOCAL / name / cls / f"{i:07d}.jpg"
        try:
            save_img(open_hf_image(r["image"]), out)
        except Exception:
            continue
        counts[cls] += 1
        rows.append(row(out, name, crop, part, HEALTHY, cls, "lab", group_hint=f"f360:{variety}", white_bg=True))
    finish(name, rows, unmapped, "; ".join(errors))


def leaf_id(stem):
    s = stem.split("___")[-1].replace("_final_masked", "")
    s = re.sub(r"\s*\(\d+\)$|\s+copy$", "", s)
    return re.sub(r"\.\d+$", "", s).strip().lower()


def ingest_plantvillage(spec):
    name = spec["name"]
    pv = RAW / "plantvillage"
    if not (pv / "raw/color").exists():
        run(f"git clone --depth 1 --filter=blob:none --sparse {spec['repo']} {pv}")
        folders = ["raw/color", "raw/segmented"]
        if DRY_RUN:
            folders = [f"raw/{k}/{c}" for k in ("color", "segmented") for c in ("Apple___healthy", "Tomato___Late_blight")]
        run(f"cd {pv} && git sparse-checkout set " + " ".join(f'"{f}"' for f in folders))
    seg = {(p.parent.name, p.stem.replace("_final_masked", "")): p for p in images_in(pv / "raw/segmented")}
    rows, unmapped = [], Counter()
    for cdir in sorted(d for d in (pv / "raw/color").iterdir() if d.is_dir()):
        if cdir.name not in PV_MAP:
            unmapped[cdir.name] += 1
            continue
        crop, part, cond = PV_MAP[cdir.name]
        cls = class_name(crop, part, cond)
        for i, p in enumerate(images_in(cdir)):
            if DRY_RUN and i >= DRY_PER_SOURCE // 2:
                break
            out = LOCAL / name / cls / f"{i:05d}.jpg"
            try:
                save_img(Image.open(p), out)
            except Exception:
                continue
            s = seg.get((cdir.name, p.stem))
            seg_out = ""
            if s is not None:
                seg_out = LOCAL / name / "_seg" / cls / f"{i:05d}.jpg"
                try:
                    save_img(Image.open(s), seg_out)
                except Exception:
                    seg_out = ""
            rows.append(row(out, name, crop, part, cond, cls, "lab", group_hint=f"pv:{cdir.name}:{leaf_id(p.stem)}", seg=seg_out))
    finish(name, rows, unmapped)


def plantdoc_boxes(r, W, H):
    sx, sy = W / float(r["width"]), H / float(r["height"])
    x1, y1, x2, y2 = (float(r[k]) for k in ("xmin", "ymin", "xmax", "ymax"))
    return max(0.0, x1 * sx), max(0.0, y1 * sy), min(float(W), x2 * sx), min(float(H), y2 * sy)


PLANTDOC_DIR = RAW / "plantdoc_od"


def clone_plantdoc():
    if not (PLANTDOC_DIR / "TRAIN").exists():
        run(f"git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset.git {PLANTDOC_DIR}")
    for need in ("TRAIN", "TEST", "train_labels.csv", "test_labels.csv"):
        if not (PLANTDOC_DIR / need).exists():
            raise FileNotFoundError(f"PlantDoc repo layout changed: {need} missing")


def ingest_plantdoc(spec):
    name = spec["name"]
    clone_plantdoc()
    rows, unmapped = [], Counter()
    for split_dir, csv_name, split in (("TRAIN", "train_labels.csv", ""), ("TEST", "test_labels.csv", "test")):
        frame = pd.read_csv(PLANTDOC_DIR / csv_name)
        lookup = {p.name: p for p in images_in(PLANTDOC_DIR / split_dir)}
        for k, (fname, g) in enumerate(frame.groupby("filename")):
            if DRY_RUN and k >= 15:
                break
            src = lookup.get(fname)
            if src is None:
                continue
            try:
                img = Image.open(src).convert("RGB")
            except Exception:
                continue
            W, H = img.size
            for j, (_, r) in enumerate(g.iterrows()):
                target = PLANTDOC_MAP.get(norm(r["class"]))
                try:
                    x1, y1, x2, y2 = plantdoc_boxes(r, W, H)
                except (ValueError, ZeroDivisionError):
                    target = None
                if target is None or x2 - x1 < 24 or y2 - y1 < 24:
                    unmapped[str(r["class"])] += 1
                    continue
                pw, ph = 0.1 * (x2 - x1), 0.1 * (y2 - y1)
                crop_img = img.crop((max(0, x1 - pw), max(0, y1 - ph), min(W, x2 + pw), min(H, y2 + ph)))
                cls = class_name(*target)
                out = LOCAL / name / cls / f"{split or 'train'}_{zlib.crc32(fname.encode()):08x}_{j}.jpg"
                save_img(crop_img, out)
                rows.append(row(out, name, *target, cls, "field", group_hint=f"pd:{fname}", official_split=split))
    finish(name, rows, unmapped)


def roboflow_download(slug, fmt, dest):
    from roboflow import Roboflow
    ws, proj = slug.split("/")
    project = Roboflow(api_key=ROBOFLOW_API_KEY).workspace(ws).project(proj)
    for v in range(20, 0, -1):
        try:
            project.version(v).download(fmt, location=str(dest), overwrite=True)
            return project
        except Exception:
            continue
    raise RuntimeError(f"No downloadable version for {slug}")


def ingest_roboflow(spec):
    name, rows, unmapped = spec["name"], [], Counter()
    dest = RAW / name
    from roboflow import Roboflow
    ws, proj = spec["repo"].split("/")
    project = Roboflow(api_key=ROBOFLOW_API_KEY).workspace(ws).project(proj)
    fmt = "folder" if "classification" in str(getattr(project, "type", "")) else "yolov8"
    roboflow_download(spec["repo"], fmt, dest)
    if fmt == "folder":
        for i, p in enumerate(images_in(dest)):
            if enough(rows):
                break
            res = resolve(spec, p.parent.name)
            if res is None:
                unmapped[p.parent.name] += 1
                continue
            cls = class_name(*res)
            out = LOCAL / name / cls / f"{i:06d}.jpg"
            save_img(Image.open(p), out)
            rows.append(row(out, name, *res, cls, spec["setting"], group_hint=f"rf:{p.stem.split('.rf.')[0]}"))
    else:
        import yaml
        names = yaml.safe_load(open(dest / "data.yaml"))["names"]
        names = names if isinstance(names, list) else [names[k] for k in sorted(names)]
        for i, p in enumerate(images_in(dest)):
            if enough(rows):
                break
            lab = label_path(p)
            if not lab.exists():
                continue
            img = Image.open(p).convert("RGB")
            W, H = img.size
            for j, line in enumerate(lab.read_text().splitlines()):
                parts = line.split()
                if len(parts) < 5:
                    continue
                c, cx, cy, w, h = int(parts[0]), *map(float, parts[1:5])
                res = resolve(spec, names[c])
                if res is None:
                    unmapped[names[c]] += 1
                    continue
                x1, y1, x2, y2 = (cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H
                if x2 - x1 < 24 or y2 - y1 < 24:
                    continue
                cls = class_name(*res)
                out = LOCAL / name / cls / f"{i:06d}_{j}.jpg"
                save_img(img.crop((max(0, x1), max(0, y1), min(W, x2), min(H, y2))), out)
                rows.append(row(out, name, *res, cls, spec["setting"], group_hint=f"rf:{p.stem.split('.rf.')[0]}"))
    finish(name, rows, unmapped)


def label_path(image_path):
    # YOLO layout: <split>/images/x.jpg -> <split>/labels/x.txt
    p = Path(image_path)
    return p.parent.parent / "labels" / f"{p.stem}.txt"


def hf_yolo_files(repo, dest, max_images, with_labels=True):
    # Downloads at most max_images image(+label) files one by one with backoff, avoiding HF rate limits on huge snapshots
    from huggingface_hub import HfApi, hf_hub_download
    files = HfApi().list_repo_files(repo, repo_type="dataset")
    imgs = [f for f in files if "/images/" in f and Path(f).suffix.lower() in IMG_EXT]
    random.Random(SEED).shuffle(imgs)
    labels = set(files)
    got = []
    for f in imgs:
        if len(got) >= max_images:
            break
        lab = str(Path(f).parent.parent / "labels" / f"{Path(f).stem}.txt").replace("\\", "/")
        if with_labels and lab not in labels:
            continue
        for attempt in range(5):
            try:
                local = hf_hub_download(repo, f, repo_type="dataset", local_dir=str(dest))
                if with_labels:
                    hf_hub_download(repo, lab, repo_type="dataset", local_dir=str(dest))
                got.append(Path(local))
                break
            except Exception as e:
                if attempt == 4:
                    print(f"WARNING {repo}: giving up on {f}: {e!r}")
                time.sleep(2 ** attempt)
    return got


def fetch(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "agritech-training-notebook"})
    return urllib.request.urlopen(req, timeout=timeout).read()


OI_TABLE = None


def openimages_table():
    # Validation images whose listed license is CC BY 2.0, joined with their verified image-level labels
    global OI_TABLE
    if OI_TABLE is None:
        imgs = pd.read_csv(download(OI_IMAGES_CSV, RAW / "oi_val_images.csv"),
                           usecols=["ImageID", "OriginalLandingURL", "License", "Author"])
        imgs = imgs[imgs.License.astype(str).str.contains("licenses/by/2.0", regex=False)]
        names = pd.read_csv(download(OI_CLASSES_CSV, RAW / "oi_classes.csv"))
        labels = pd.read_csv(download(OI_LABELS_CSV, RAW / "oi_val_labels.csv"), usecols=["ImageID", "LabelName", "Confidence"])
        labels = labels[pd.to_numeric(labels.Confidence, errors="coerce") >= 1].merge(names, on="LabelName")
        text = labels.groupby("ImageID").DisplayName.apply(lambda s: " | ".join(map(str, s)).lower()).rename("labels")
        OI_TABLE = imgs.set_index("ImageID").join(text, how="inner")
        print(f"Open Images: {len(OI_TABLE)} CC BY 2.0 validation images with labels")
    return OI_TABLE


def ingest_openimages(spec):
    name, cap = spec["name"], (DRY_PER_SOURCE if DRY_RUN else spec["cap"])
    t = openimages_table()
    crop_hit = t.labels.str.contains(rf"\b(?:{CROP_WORDS})s?\b", regex=True)
    flora_hit = t.labels.str.contains(rf"\b(?:{FLORA_WORDS})s?\b", regex=True)
    pick = t[~crop_hit & ~flora_hit] if spec["mode"] == "objects" else t[flora_hit & ~crop_hit]
    pick = pick.sample(frac=1.0, random_state=SEED).head(int(cap * 1.3))
    print(f"[{name}] {len(pick)} candidate images")

    def get_one(item):
        image_id, r = item
        out = LOCAL / name / f"{image_id}.jpg"
        try:
            save_img(Image.open(io.BytesIO(fetch(OI_IMAGE_URL.format(image_id)))), out)
            return neg_row(out, name, attribution=f"{r.Author} | {r.OriginalLandingURL} | CC BY 2.0")
        except Exception:
            return None

    with ThreadPoolExecutor(32) as ex:
        rows = [r for r in ex.map(get_one, pick.iterrows()) if r is not None][:cap]
    finish(name, rows, Counter(), "" if len(rows) >= 0.5 * min(cap, len(pick)) else f"only {len(rows)} of {len(pick)} downloads worked")


def ingest_polyhaven(spec):
    name = spec["name"]
    assets = json.loads(fetch("https://api.polyhaven.com/assets?type=textures"))
    veg = re.compile(r"leaf|leaves|grass|moss|plant|forest|foliage|vegetation|flower|crop|hay|straw|field")
    ids = [k for k, v in assets.items() if not veg.search(" ".join(v.get("tags", []) + v.get("categories", []) + [k]).lower())]
    random.Random(SEED).shuffle(ids)
    per = spec["crops_per_texture"]
    ids = ids[: (3 if DRY_RUN else spec["cap"] // per)]
    rows, failed_ids = [], []
    rng = random.Random(SEED)
    for tid in ids:
        try:
            url = json.loads(fetch(f"https://api.polyhaven.com/files/{tid}"))["Diffuse"]["1k"]["jpg"]["url"]
            img = Image.open(io.BytesIO(fetch(url))).convert("RGB")
        except Exception as e:
            failed_ids.append(tid)
            continue
        W, H = img.size
        for k in range(per):
            s = rng.randint(min(W, H) // 4, min(W, H))
            x, y = rng.randint(0, W - s), rng.randint(0, H - s)
            out = LOCAL / name / f"{tid}_{k}.jpg"
            save_img(img.crop((x, y, x + s, y + s)), out)
            rows.append(neg_row(out, name, hint=f"ph:{tid}", attribution=f"Poly Haven texture {tid} | CC0"))
    finish(name, rows, Counter(), f"{len(failed_ids)} textures failed" if len(failed_ids) > len(ids) // 2 else "")


def ingest_negative(spec):
    name, rows, errors = spec["name"], [], []
    cap = DRY_PER_SOURCE if DRY_RUN else spec["cap"]
    if spec["kind"] == "openimages":
        return ingest_openimages(spec)
    if spec["kind"] == "polyhaven":
        return ingest_polyhaven(spec)
    if spec["kind"] == "hf_yolo_images":
        files = hf_yolo_files(spec["repo"], RAW / name, cap, with_labels=False)
        for i, p in enumerate(files[:cap]):
            out = LOCAL / name / f"{i:06d}.jpg"
            try:
                save_img(Image.open(p), out)
                rows.append(neg_row(out, name, hint=f"rf:{p.stem.split('.rf.')[0]}", attribution=f"{spec['repo']} | {spec['license']}"))
            except Exception:
                pass
    else:
        for i, (split, r) in enumerate(hf_rows(spec["repo"], spec.get("config"), errors)):
            if len(rows) >= cap or (DRY_RUN and i >= DRY_SCAN_ROWS * 20):
                break
            if spec.get("only_label") and r.get("label") != spec["only_label"]:
                continue
            out = LOCAL / name / f"{i:07d}.jpg"
            try:
                save_img(open_hf_image(r["image"]), out)
            except Exception:
                continue
            rows.append(neg_row(out, name, attribution=f"{spec['repo']} | {spec['license']}"))
    finish(name, rows, Counter(), "; ".join(errors))


INGEST = {"hf": ingest_hf, "zip_keywords": ingest_zip_keywords, "fruits360": ingest_fruits360,
          "plantvillage": ingest_plantvillage, "plantdoc": ingest_plantdoc, "roboflow": ingest_roboflow}

for spec in SOURCES + NEGATIVES:
    name = spec["name"]
    if restore(name):
        print(f"[{name}] restored")
        continue
    if spec["kind"] == "roboflow" and not ROBOFLOW_API_KEY:
        set_status(name, "SKIPPED", note="no ROBOFLOW_API_KEY secret")
        continue
    t0 = time.time()
    shutil.rmtree(LOCAL / name, ignore_errors=True)
    (LOCAL / name).mkdir(parents=True, exist_ok=True)
    try:
        (ingest_negative if name.startswith("neg_") else INGEST[spec["kind"]])(spec)
    except Exception as e:
        shutil.rmtree(LOCAL / name, ignore_errors=True)
        set_status(name, "FAILED", note=repr(e))
    print(f"[{name}] {STATUS.get(name, {}).get('status')} in {(time.time() - t0) / 60:.1f} min")

print()
ingest_status = status_table()

## 3. Deduplicate, group near-duplicates, build splits

- Exact and resized duplicates are removed with a rotation/flip-invariant perceptual hash. If one image appears under two different classes, every copy is dropped.
- Near-duplicates (same leaf photographed again, zoomed or augmented) are grouped with CNN embeddings (cosine >= 0.95), and a group never crosses splits.
- **Test split:** a whole field source is held out per class when the class has 2 or more field sources (cross-source test). Otherwise 15% of groups are held out. PlantDoc uses its official test split. Lab sources are used for train and validation only.
- After splitting, any test image that is still near-identical (cosine >= 0.97) to a training image is removed. The leak count is reported.

In [ ]:
import imagehash
from multiprocessing import Pool

frames = []
for spec in SOURCES + NEGATIVES:
    m = LOCAL / spec["name"] / "manifest.csv"
    if m.exists():
        frames.append(pd.read_csv(m, keep_default_na=False))
if not frames:
    raise RuntimeError("No source produced any images. Check the status table from step 2.")
df = pd.concat(frames, ignore_index=True)
for col in ("attribution", "seg", "group_hint", "official_split"):
    df[col] = df[col].astype(str) if col in df else ""
if df.cls.eq("unknown").all() or not df.cls.eq("unknown").any():
    print("WARNING: need both plant classes and unknown negatives; check the status table")
df["license"] = df.source.map(lambda s: SPEC[s]["license"] if s in SPEC else next(n["license"] for n in NEGATIVES if n["name"] == s))
print(len(df), "images before dedup")


def dihedral_phash(path):
    try:
        img = Image.open(path).convert("L").resize((64, 64))
    except Exception:
        return -1
    best = None
    for flip in (False, True):
        im = ImageOps.mirror(img) if flip else img
        for rot in (0, 90, 180, 270):
            h = int(str(imagehash.phash(im.rotate(rot))), 16)
            best = h if best is None else min(best, h)
    return best


hash_cache = ROOT / "phash.parquet"
cached = pd.read_parquet(hash_cache) if hash_cache.exists() else pd.DataFrame(columns=["path", "phash"])
todo = sorted(set(df.path) - set(cached.path))
if todo:
    with Pool(os.cpu_count()) as pool:
        hashes = pool.map(dihedral_phash, todo, chunksize=256)
    cached = pd.concat([cached, pd.DataFrame({"path": todo, "phash": [str(h) for h in hashes]})])
    cached.to_parquet(hash_cache)
df = df.merge(cached, on="path")
df = df[df.phash != "-1"]

conflict = df.groupby("phash").cls.nunique()
conflict_hashes = set(conflict[conflict > 1].index)
setting_rank = {"field": 0, "mixed": 1, "lab": 2, "neg": 3}
df["_rank"] = df.setting.map(setting_rank)
n0 = len(df)
dropped_conflict = int(df.phash.isin(conflict_hashes).sum())
df = df[~df.phash.isin(conflict_hashes)].sort_values(["_rank", "official_split"], ascending=[True, False])
df = df.drop_duplicates("phash", keep="first").drop(columns="_rank").reset_index(drop=True)
print(f"Removed {n0 - len(df)} duplicates ({dropped_conflict} because the same image had different labels)")

In [ ]:
import torch, timm
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

device = "cuda"
emb_cache = ROOT / "embeddings.npz"
emb_tf = T.Compose([T.Resize((160, 160)), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])


class PathDS(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            return emb_tf(Image.open(self.paths[i]).convert("RGB"))
        except Exception:
            return torch.zeros(3, 160, 160)


known = {}
if emb_cache.exists():
    z = np.load(emb_cache, allow_pickle=True)
    known = dict(zip(z["paths"], z["emb"]))
todo = [p for p in df.path if p not in known]
if todo:
    embedder = timm.create_model("efficientnet_b0.ra_in1k", pretrained=True, num_classes=0).to(device).eval()
    out = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for x in DataLoader(PathDS(todo), batch_size=256, num_workers=os.cpu_count()):
            out.append(F.normalize(embedder(x.to(device)).float(), dim=1).half().cpu().numpy())
    known.update(zip(todo, np.concatenate(out)))
    np.savez(emb_cache, paths=np.array(list(known)), emb=np.stack(list(known.values())))
    del embedder
EMB = np.stack([known[p] for p in df.path]).astype(np.float16)
df["emb_row"] = np.arange(len(df))
df["white_bg"] = df.white_bg.astype(str).str.lower().eq("true")


class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]
            a = self.p[a]
        return a

    def union(self, a, b):
        self.p[self.find(a)] = self.find(b)


# Near-duplicate groups inside each (source, class)
group = np.empty(len(df), dtype=object)
for (src, cls), idx in df.groupby(["source", "cls"]).indices.items():
    uf = UnionFind(len(idx))
    e = torch.from_numpy(EMB[idx]).to(device)
    for s in range(0, len(idx), 2048):
        sim = e[s:s + 2048] @ e.T
        a, b = torch.nonzero(sim >= NEAR_DUP_SIM, as_tuple=True)
        for i, j in zip((a + s).tolist(), b.tolist()):
            if i < j:
                uf.union(i, j)
    hints = df.group_hint.values[idx]
    first = {}
    for k, h in enumerate(hints):
        if h:
            if h in first:
                uf.union(k, first[h])
            else:
                first[h] = k
    for k in range(len(idx)):
        group[idx[k]] = f"{src}|{cls}|{uf.find(k)}"
df["group"] = group
print(df.groupby("setting").group.nunique(), "unique groups per setting")

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

rng = np.random.RandomState(SEED)

# Cap very large (source, class) pools by sampling whole groups
keep = []
for (src, cls), g in df.groupby(["source", "cls"]):
    if len(g) <= CAP_PER_SOURCE_CLASS:
        keep.append(g)
        continue
    groups = g.group.unique()
    rng.shuffle(groups)
    sizes = g.group.value_counts()
    chosen, total = [], 0
    for gr in groups:
        if total >= CAP_PER_SOURCE_CLASS:
            break
        chosen.append(gr)
        total += sizes[gr]
    keep.append(g[g.group.isin(chosen)])
df = pd.concat(keep, ignore_index=True)

counts = df.groupby("cls").size()
crop_of_cls = df.drop_duplicates("cls").set_index("cls").crop
min_needed = counts.index.map(lambda c: min(MIN_IMAGES, MIN_IMAGES_BY_CROP.get(crop_of_cls[c], MIN_IMAGES)))
dropped_small = counts[(counts < min_needed) & (counts.index != "unknown")]
df = df[~df.cls.isin(dropped_small.index)].reset_index(drop=True)
print(f"Dropped {len(dropped_small)} classes under {MIN_IMAGES} images:")
print(dropped_small.sort_values().to_string())


def gsplit(g, frac):
    # Returns (held_out_index, rest_index), never splitting a near-duplicate group
    if g.group.nunique() < 2:
        return g.index[:0], g.index
    rest, held = next(GroupShuffleSplit(1, test_size=frac, random_state=SEED).split(g, groups=g.group))
    return g.index[held], g.index[rest]


def assign_val_train(g):
    val, train = gsplit(g, 0.15)
    df.loc[val, "split"] = "val"
    df.loc[train, "split"] = "train"


df["split"] = ""
df["test_kind"] = ""
df.loc[df.official_split == "test", ["split", "test_kind"]] = ["test", "official"]
for cls, g in df[df.split == ""].groupby("cls"):
    if cls == "unknown":
        test, rest = gsplit(g, 0.1)
        df.loc[test, ["split", "test_kind"]] = ["test", "negatives"]
        assign_val_train(g.loc[rest])
        continue
    field_sources = g[g.setting == "field"].groupby("source").size()
    eligible = field_sources[(field_sources >= MIN_TEST) & (field_sources <= 0.5 * len(g))]
    held = None
    if len(field_sources) >= 2 and len(eligible):
        held = (eligible - 0.2 * len(g)).abs().idxmin()
        df.loc[g.index[g.source == held], ["split", "test_kind"]] = ["test", "cross_source"]
    for src, gs in g[g.source != held].groupby("source"):
        if gs.setting.iloc[0] in ("field", "mixed") and held is None:
            test, rest = gsplit(gs, 0.15)
            df.loc[test, ["split", "test_kind"]] = ["test", "in_source"]
            assign_val_train(gs.loc[rest])
        else:
            assign_val_train(gs)

# Leak check: drop test images that are near-identical to any training image
train_idx = np.where(df.split == "train")[0]
test_idx = np.where(df.split == "test")[0]
leaks = []
if len(train_idx) and len(test_idx):
    tr = torch.from_numpy(EMB[df.emb_row.values[train_idx]]).to(device)
    for s in range(0, len(test_idx), 1024):
        te = torch.from_numpy(EMB[df.emb_row.values[test_idx[s:s + 1024]]]).to(device)
        m = (te @ tr.T).max(dim=1).values
        leaks.extend(test_idx[s:s + 1024][(m >= LEAK_SIM).cpu().numpy()])
    del tr
df = df.drop(index=df.index[leaks]).reset_index(drop=True)
print(f"Leak check removed {len(leaks)} test images")

test_counts = df[df.split == "test"].groupby("cls").size()
no_test = sorted(set(df.cls) - set(test_counts[test_counts >= MIN_TEST].index))
print(f"{len(no_test)} classes have fewer than {MIN_TEST} test images (reported, but kept for training):", no_test)

CLASSES = sorted(c for c in df.cls.unique() if c != "unknown") + ["unknown"]
CLS_IDX = {c: i for i, c in enumerate(CLASSES)}
df["label"] = df.cls.map(CLS_IDX)
df.to_parquet(ROOT / "dataset_v2.parquet")
print(len(CLASSES), "classes")
print(df.groupby(["split", "setting"]).size().unstack(fill_value=0))
print(df[df.split == "test"].groupby("test_kind").size())

## 4. Augmentation and loaders

Lab images get random real backgrounds (PlantVillage via its segmentation masks, white-background fruit photos via a brightness mask). Training also uses blur, lighting, crops, CutMix and MixUp.
Sampling is class-balanced (inverse square-root frequency), with field images weighted 2x and `unknown` at about 8% of each batch.

In [ ]:
import cv2
from torch.utils.data import WeightedRandomSampler
from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy
from timm.utils import ModelEmaV3
from tqdm.auto import tqdm

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
BG = df[(df.cls == "unknown") & (df.split == "train") & df.source.isin(["neg_openimages_objects", "neg_deepweeds", "neg_polyhaven_textures"])].path.tolist()

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.3, 1.0), ratio=(0.7, 1.4)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(30),
    T.RandomApply([T.ColorJitter(0.4, 0.4, 0.35, 0.05)], p=0.8),
    T.RandomApply([T.GaussianBlur(9, sigma=(0.1, 3.0))], p=0.3),
    T.RandomGrayscale(p=0.05), T.RandomAdjustSharpness(2, p=0.2),
    T.ToTensor(), T.Normalize(MEAN, STD), T.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])


def paste_on_background(rgb, mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours or not BG:
        return Image.fromarray(rgb)
    filled = np.zeros_like(mask)
    cv2.drawContours(filled, [max(contours, key=cv2.contourArea)], -1, 1, thickness=-1)
    alpha = cv2.GaussianBlur(filled.astype(np.float32), (5, 5), 0)[..., None]
    h, w = mask.shape
    bg = T.RandomResizedCrop((h, w), scale=(0.2, 1.0))(Image.open(random.choice(BG)).convert("RGB"))
    return Image.fromarray((rgb * alpha + np.asarray(bg, np.float32) * (1 - alpha)).astype(np.uint8))


def composite(row):
    if row["seg"]:
        rgb = np.asarray(Image.open(row["seg"]).convert("RGB"))
        return paste_on_background(rgb, (rgb.max(axis=2) > 20).astype(np.uint8))
    rgb = np.asarray(Image.open(row["path"]).convert("RGB"))
    return paste_on_background(rgb, (rgb.min(axis=2) < 235).astype(np.uint8))


class LeafDS(Dataset):
    def __init__(self, frame, tf, composite_p=0.0):
        self.rows = frame[["path", "seg", "white_bg", "label"]].to_dict("records")
        self.tf, self.p = tf, composite_p

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        try:
            if self.p and (r["seg"] or r["white_bg"]) and random.random() < self.p:
                img = composite(r)
            else:
                img = Image.open(r["path"]).convert("RGB")
        except Exception:
            img = Image.open(r["path"]).convert("RGB")
        return self.tf(img), r["label"]


train_df = df[df.split == "train"].reset_index(drop=True)
val_df = df[df.split == "val"].reset_index(drop=True)
test_df = df[df.split == "test"].reset_index(drop=True)

cls_count = train_df.cls.value_counts()
w = train_df.cls.map(lambda c: cls_count[c] ** -0.5).to_numpy() * np.where(train_df.setting == "field", 2.0, 1.0)
unk = (train_df.cls == "unknown").to_numpy()
if unk.any() and (~unk).any():
    w[unk] *= 0.08 * w[~unk].sum() / (0.92 * w[unk].sum())
assert len(train_df) and EPOCH_SIZE >= BATCH, "Training set is empty; check the status table and split summary"
sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), num_samples=EPOCH_SIZE, replacement=True)

workers = os.cpu_count() or 2
kw = dict(batch_size=BATCH, num_workers=workers, pin_memory=True, persistent_workers=True)
train_loader = DataLoader(LeafDS(train_df, train_tf, composite_p=0.5), sampler=sampler, drop_last=True, **kw)
val_loader = DataLoader(LeafDS(val_df, eval_tf), shuffle=False, **kw)
test_loader = DataLoader(LeafDS(test_df, eval_tf), shuffle=False, **kw)
NC = len(CLASSES)
print(f"train {len(train_df)}, val {len(val_df)}, test {len(test_df)}, classes {NC}")

## 5. Train with resume (head first, then full network)

A checkpoint is written to Drive after every epoch. Re-running this cell continues from the last finished epoch.

In [ ]:
model = timm.create_model(BACKBONE, pretrained=True, num_classes=NC).to(device)
ema = ModelEmaV3(model, decay=0.9998)
scaler = torch.amp.GradScaler("cuda")
mixup = Mixup(mixup_alpha=0.2, cutmix_alpha=1.0, prob=0.6, switch_prob=0.5, label_smoothing=0.1, num_classes=NC)
soft_ce = SoftTargetCrossEntropy()
hard_ce = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
SCHEDULE = [("head", e) for e in range(HEAD_EPOCHS)] + [("full", e) for e in range(FULL_EPOCHS)]
LAST, BEST = CKPT / "classifier_last.pth", CKPT / "classifier_best.pth"
UNK = CLS_IDX["unknown"]


def make_opt(stage):
    head = list(model.get_classifier().parameters())
    for p in model.parameters():
        p.requires_grad = stage == "full"
    for p in head:
        p.requires_grad = True
    params = [p for p in model.parameters() if p.requires_grad]
    lr = LR_HEAD if stage == "head" else LR_FULL
    n = HEAD_EPOCHS if stage == "head" else FULL_EPOCHS
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.05 if stage == "full" else 1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=n * len(train_loader), pct_start=0.05 if stage == "full" else 0.1)
    return opt, sched


def predict(net, loader):
    net.eval()
    out, ys = [], []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for x, y in loader:
            out.append(net(x.to(device, non_blocking=True)).float().cpu())
            ys.append(y)
    if not out:
        return torch.zeros(0, NC), torch.zeros(0, dtype=torch.long)
    return torch.cat(out), torch.cat(ys)


def val_score(logits, labels, frame):
    pred = logits.argmax(1).numpy()
    y = labels.numpy()
    field = (frame.setting == "field").to_numpy()
    known = y != UNK
    recalls = [(pred[y == c] == c).mean() for c in np.unique(y[known])]
    return dict(acc=float((pred == y).mean()) if len(y) else 0.0,
                field_acc=float((pred[field] == y[field]).mean()) if field.any() else 0.0,
                macro_recall=float(np.mean(recalls)) if recalls else 0.0,
                unknown_recall=float((pred[~known] == UNK).mean()) if (~known).any() else 0.0)


state = dict(step=0, history=[], best=-1.0)
if LAST.exists():
    ck = torch.load(LAST, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"])
    ema.module.load_state_dict(ck["ema"])
    state = ck["state"]
    print(f"Resuming at schedule step {state['step']} of {len(SCHEDULE)}")

opt = sched = None
current_stage = None
while state["step"] < len(SCHEDULE):
    stage, ep = SCHEDULE[state["step"]]
    if stage != current_stage:
        opt, sched = make_opt(stage)
        current_stage = stage
        if LAST.exists():
            ck = torch.load(LAST, map_location=device, weights_only=False)
            if ck.get("stage") == stage and ck.get("opt"):
                opt.load_state_dict(ck["opt"])
                sched.load_state_dict(ck["sched"])
                scaler.load_state_dict(ck["scaler"])
    model.train()
    total, n = 0.0, 0
    for x, y in tqdm(train_loader, desc=f"{stage} {ep + 1}", leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            if stage == "full":
                xm, ym = mixup(x, y)
                loss = soft_ce(model(xm), ym)
            else:
                loss = hard_ce(model(x), y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        ema.update(model)
        total, n = total + loss.item() * x.size(0), n + x.size(0)
    metrics = val_score(*predict(ema.module, val_loader), val_df)
    score = 0.5 * metrics["field_acc"] + 0.3 * metrics["macro_recall"] + 0.2 * metrics["unknown_recall"]
    state["history"].append(dict(stage=stage, epoch=ep + 1, loss=total / n, score=score, **metrics))
    print(f"{stage} {ep + 1}: loss {total / n:.3f} | val acc {metrics['acc']:.3f} field {metrics['field_acc']:.3f} "
          f"macro-recall {metrics['macro_recall']:.3f} unknown {metrics['unknown_recall']:.3f}")
    if score > state["best"]:
        state["best"] = score
        torch.save({"model": ema.module.state_dict(), "classes": CLASSES}, BEST)
    state["step"] += 1
    torch.save({"model": model.state_dict(), "ema": ema.module.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict(), "stage": stage, "state": state}, LAST)
print("Training finished. Best validation score:", round(state["best"], 4))

## 6. Calibration and honest evaluation

Temperature is fitted on validation data. The threshold is the lowest confidence at which validation precision reaches the target. Test numbers are reported by test type (cross-source, in-source, official PlantDoc) and never used to choose anything.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

best_model = timm.create_model(BACKBONE, pretrained=False, num_classes=NC).to(device)
best_model.load_state_dict(torch.load(BEST, map_location=device, weights_only=False)["model"])
val_logits, val_y = predict(best_model, val_loader)
test_logits, test_y = predict(best_model, test_loader)

log_t = torch.zeros(1, requires_grad=True)
lbfgs = torch.optim.LBFGS([log_t], lr=0.05, max_iter=300)


def closure():
    lbfgs.zero_grad()
    loss = F.cross_entropy(val_logits / log_t.exp(), val_y)
    loss.backward()
    return loss


if len(val_y):
    lbfgs.step(closure)
TEMPERATURE = float(log_t.exp()) if len(val_y) and math.isfinite(float(log_t.exp())) else 1.0
if not 0.5 <= TEMPERATURE <= 10.0:
    print(f"WARNING: fitted temperature {TEMPERATURE:.2f} is implausible (model barely trained?); clamping to [0.5, 10]")
    TEMPERATURE = float(min(max(TEMPERATURE, 0.5), 10.0))


def ece(probs, y, bins=15):
    conf, pred = probs.max(1)
    edges = torch.linspace(0, 1, bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.float().mean() * ((pred[m] == y[m]).float().mean() - conf[m].mean()).abs()
    return float(e)


val_p = torch.softmax(val_logits / TEMPERATURE, 1)
test_p = torch.softmax(test_logits / TEMPERATURE, 1)
ECE_BEFORE = ece(torch.softmax(test_logits, 1), test_y)
ECE_AFTER = ece(test_p, test_y)

# Threshold: lowest confidence where answered plant predictions on validation reach TARGET_PRECISION
vc, vp = val_p.max(1)
vy = val_y
rows = []
for t in np.round(np.arange(0.3, 0.96, 0.05), 2):
    ans = (vc >= t) & (vp != UNK)
    plant = vy != UNK
    rows.append(dict(threshold=t, precision=float((vp[ans] == vy[ans]).float().mean()) if ans.any() else 0.0,
                     coverage=float(ans[plant].float().mean())))
thr_table = pd.DataFrame(rows)
ok = thr_table[thr_table.precision >= TARGET_PRECISION]
THRESHOLD = float(ok.threshold.iloc[0]) if len(ok) else 0.75
print(f"Temperature {TEMPERATURE:.3f} | test ECE {ECE_BEFORE:.3f} -> {ECE_AFTER:.3f} | recommended threshold {THRESHOLD}")
print(thr_table.round(3).to_string(index=False))

tc, tp = test_p.max(1)
tp, tc, ty = tp.numpy(), tc.numpy(), test_y.numpy()
REPORT = {"classes": NC, "temperature": TEMPERATURE, "ece_before": ECE_BEFORE, "ece_after": ECE_AFTER, "threshold": THRESHOLD}
for kind in ["cross_source", "in_source", "official", "negatives"]:
    m = (test_df.test_kind == kind).to_numpy()
    if m.any():
        acc = float((tp[m] == ty[m]).mean())
        ans = m & (tc >= THRESHOLD) & (tp != UNK)
        REPORT[f"test_{kind}"] = dict(n=int(m.sum()), top1=acc,
                                      precision_at_threshold=float((tp[ans] == ty[ans]).mean()) if ans.any() else None,
                                      coverage_at_threshold=float(ans[m].mean()))
field_m = (test_df.setting == "field").to_numpy()
REPORT["test_field_top1"] = float((tp[field_m] == ty[field_m]).mean())
lab_v = (val_df.setting == "lab").to_numpy()
REPORT["val_lab_top1"] = float((val_logits.argmax(1).numpy()[lab_v] == val_y.numpy()[lab_v]).mean()) if lab_v.any() else None
print(json.dumps(REPORT, indent=1))

present = sorted(set(ty))
if present:
    cr = classification_report(ty, tp, labels=present, target_names=[CLASSES[i] for i in present], output_dict=True, zero_division=0)
    per_class = pd.DataFrame(cr).T.drop(index=["accuracy", "macro avg", "weighted avg", "micro avg"], errors="ignore")
else:
    per_class = pd.DataFrame(columns=["precision", "recall", "f1-score", "support"])
per_class.to_csv(EXPORT / "per_class_test.csv")
weak = per_class[(per_class["recall"] < 0.6) | (per_class["support"] < MIN_TEST)].sort_values("recall")
print("Weak classes (test recall < 0.6 or support < 20):")
print(weak.round(3).to_string())

cm = confusion_matrix(ty, tp, labels=list(range(NC)))
pairs = [(cm[i, j], CLASSES[i], CLASSES[j]) for i in range(NC) for j in range(NC) if i != j and cm[i, j] > 0]
top_confusions = sorted(pairs, reverse=True)[:25]
print("Most frequent confusions (true -> predicted):")
for n_, a, b in top_confusions:
    print(f"  {n_:4d}  {a} -> {b}")

crop_of = np.array([c.split("___")[0].rsplit("_", 1)[0] if c != "unknown" else "unknown" for c in CLASSES])
test_crop = crop_of[ty]
crop_acc = pd.Series({crop: float((tp[test_crop == crop] == ty[test_crop == crop]).mean()) for crop in sorted(set(test_crop))})
print("Test top-1 by crop:")
print(crop_acc.round(3).to_string())

hist = pd.DataFrame(state["history"])
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(hist.loss)
ax[0].set_title("Training loss")
for col in ["field_acc", "macro_recall", "unknown_recall", "acc"]:
    ax[1].plot(hist[col], label=col)
ax[1].legend()
ax[1].set_title("Validation")
plt.tight_layout()
plt.savefig(FIGS / "training_curves.png", dpi=150)
plt.show()

cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
plt.figure(figsize=(30, 28))
plt.imshow(cmn, cmap="Greens", vmin=0, vmax=1)
plt.xticks(range(NC), CLASSES, rotation=90, fontsize=5)
plt.yticks(range(NC), CLASSES, fontsize=5)
plt.title("Test confusion matrix (row-normalised)")
plt.tight_layout()
plt.savefig(FIGS / "confusion_matrix_test.png", dpi=150)
plt.close()

## 7. Export the classifier (ONNX opset 17, `input` -> `logits`, temperature baked in)

In [ ]:
import onnxruntime as ort


class Calibrated(torch.nn.Module):
    def __init__(self, m, t):
        super().__init__()
        self.m = m
        self.register_buffer("t", torch.tensor(t, dtype=torch.float32))

    def forward(self, x):
        return self.m(x) / self.t


export_model = Calibrated(best_model.float().cpu().eval(), TEMPERATURE).eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
raw_onnx = CKPT / "classifier_raw.onnx"
kwargs = dict(opset_version=17, input_names=["input"], output_names=["logits"])
try:
    torch.onnx.export(export_model, dummy, str(raw_onnx), dynamo=False, **kwargs)
except TypeError:
    torch.onnx.export(export_model, dummy, str(raw_onnx), **kwargs)
CLS_ONNX = EXPORT / "mobilenetv4_plant_disease.onnx"
try:
    import onnxslim
    onnxslim.slim(str(raw_onnx), str(CLS_ONNX))
except Exception as e:
    print(f"WARNING: onnxslim failed ({e!r}); exporting the unslimmed model instead")
    shutil.copy(raw_onnx, CLS_ONNX)

size_mb = CLS_ONNX.stat().st_size / 1e6
sess = ort.InferenceSession(str(CLS_ONNX), providers=["CPUExecutionProvider"])
assert sess.get_inputs()[0].name == "input" and list(sess.get_inputs()[0].shape) == [1, 3, IMG_SIZE, IMG_SIZE]
assert sess.get_outputs()[0].name == "logits" and sess.get_outputs()[0].shape[-1] == NC
diff = np.abs(sess.run(None, {"input": dummy.numpy()})[0] - export_model(dummy).detach().numpy()).max()
assert diff < 1e-3, diff


def bench(path, shape, threads, runs=50):
    o = ort.SessionOptions()
    o.intra_op_num_threads = threads
    s = ort.InferenceSession(str(path), o, providers=["CPUExecutionProvider"])
    x = np.random.rand(*shape).astype(np.float32)
    name = s.get_inputs()[0].name
    for _ in range(5):
        s.run(None, {name: x})
    t0 = time.perf_counter()
    for _ in range(runs):
        s.run(None, {name: x})
    return (time.perf_counter() - t0) / runs * 1000


REPORT["classifier_onnx_mb"] = size_mb
REPORT["classifier_cpu_ms_1thread"] = bench(CLS_ONNX, (1, 3, IMG_SIZE, IMG_SIZE), 1)
REPORT["classifier_cpu_ms_4threads"] = bench(CLS_ONNX, (1, 3, IMG_SIZE, IMG_SIZE), 4)
print(f"Classifier ONNX {size_mb:.1f} MB, max diff {diff:.2e}, CPU {REPORT['classifier_cpu_ms_1thread']:.1f} ms (1 thread)")
if size_mb > MAX_ONNX_MB:
    print(f"WARNING: model is larger than {MAX_ONNX_MB} MB. Use a smaller BACKBONE for the browser model.")

## 8. Detector: leaf, fruit, ear, pest (YOLO11n)

Boxes from each source are mapped to 4 classes. Every source labels only its own classes, so a pseudo-labeling pass adds confident boxes for the missing classes before a second training round.
mAP for each class is measured only on test images from sources that label that class, so a correctly found leaf on a fruit-only photo is not counted as an error.

In [ ]:
DET_NAMES = {0: "leaf", 1: "fruit", 2: "ear", 3: "pest"}
DET_SOURCES = [
    dict(name="det_tea", repo="Project-AgML/tea_leaf_disease_pest_detection", map="all:0"),
    dict(name="det_truss_tomato", repo="Project-AgML/truss_tomato_detection_greenhouse", map={"mature": 1, "raw": 1}),
    dict(name="det_coffee", repo="Project-AgML/coffee_detection", map={"unripe": 1, "ripening": 1, "ripe": 1, "spoiled": 1}),
    dict(name="det_strawberry", repo="Project-AgML/strawberry_growth_detection",
         map={"green": 1, "overripe": 1, "red": 1, "small g": 1, "turning red": 1, "white": 1}),
    dict(name="det_grape", repo="Project-AgML/grapevine_bunch_detection", map="all:1"),
    dict(name="det_raspberry", repo="Project-AgML/raspberry_maturity_detection", map={"Unripe Berries": 1, "Ripe Berries": 1}),
    dict(name="det_litchi", repo="Project-AgML/litchi_fruit_detection", map="all:1"),
    dict(name="det_banana", repo="Project-AgML/banana_bunch_detection", map="all:1"),
    dict(name="det_date", repo="Project-AgML/date_cluster_detection", map="all:1"),
    dict(name="det_wheat_head", repo="Project-AgML/wheat_head_counting", map="all:2"),
    dict(name="det_sdau_wheat", repo="Project-AgML/SDAU_wheat_head_detection", map="all:2"),
    dict(name="det_rice_panicle", repo="Project-AgML/rice_panicle_detection", config="raw", map="all:2"),
    dict(name="det_insects", repo="LibreYOLO/insects-mytwu", kind="yolo", map="all:3"),
    dict(name="det_pests", repo="LibreYOLO/pests-2xlvx", kind="yolo", map="all:3"),
]
YOLO_DIR = Path(f"/content/yolo_v2{SUFFIX}")
for s in ("train", "val", "test"):
    (YOLO_DIR / "images" / s).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / s).mkdir(parents=True, exist_ok=True)


def det_class(spec, raw):
    m = spec["map"]
    if isinstance(m, str):
        return int(m.split(":")[1])
    return m.get(raw)


def pick_split(key):
    h = zlib.crc32(key.encode()) % 10
    return "test" if h == 0 else "val" if h == 1 else "train"


def write_det(img, boxes, split, stem):
    img = img.convert("RGB")
    W, H = img.size
    lines = []
    for c, x, y, w, h in boxes:
        x1, y1, x2, y2 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
        if x2 - x1 < 3 or y2 - y1 < 3:
            continue
        lines.append(f"{c} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}")
    if not lines:
        return False
    img.thumbnail((1024, 1024))
    img.save(YOLO_DIR / "images" / split / f"{stem}.jpg", quality=92)
    (YOLO_DIR / "labels" / split / f"{stem}.txt").write_text("\n".join(lines))
    return True


def det_category_names(features):
    objs = features["objects"]
    cats = objs["categories"] if isinstance(objs, dict) else objs.feature["categories"]
    inner = cats[0] if isinstance(cats, list) else getattr(cats, "feature", cats)
    return getattr(inner, "names", None)


def det_restore(name):
    tar = PREP_DRIVE / f"{name}.tar"
    if not tar.exists():
        return False
    try:
        with tarfile.open(tar) as t:
            t.extractall(YOLO_DIR, filter="data")
        set_status(name, "OK", len(list((YOLO_DIR / "images").rglob(f"{name}__*"))), note="restored from Drive")
        return True
    except Exception as e:
        print(f"WARNING [{name}] cached archive is unreadable ({e!r}); downloading again")
        tar.unlink(missing_ok=True)
        return False


def det_done(name, n):
    if n == 0:
        set_status(name, "EMPTY", note="no usable boxes; check the category map")
        return
    det_pack(name)
    set_status(name, "OK", n, 1)


def det_pack(name):
    tmp = PREP_DRIVE / f"{name}.tar.part"
    with tarfile.open(tmp, "w") as t:
        for p in YOLO_DIR.rglob(f"{name}__*"):
            t.add(p, arcname=str(p.relative_to(YOLO_DIR)))
    tmp.rename(PREP_DRIVE / f"{name}.tar")


def ingest_det(spec):
    name, n = spec["name"], 0
    if spec.get("kind") == "yolo":
        for p in hf_yolo_files(spec["repo"], RAW / name, YOLO_CAP_PER_SOURCE):
            lab = label_path(p)
            if not lab.exists() or n >= YOLO_CAP_PER_SOURCE:
                continue
            img = Image.open(p)
            W, H = img.size
            boxes = []
            for line in lab.read_text().splitlines():
                v = line.split()
                if len(v) >= 5:
                    cx, cy, w, h = map(float, v[1:5])
                    boxes.append((det_class(spec, None), (cx - w / 2) * W, (cy - h / 2) * H, w * W, h * H))
            n += write_det(img, boxes, pick_split(f"{name}{p.stem.split('.rf.')[0]}"), f"{name}__{n:05d}")
    else:
        from datasets import load_dataset
        ds = load_dataset(spec["repo"], spec.get("config"), split="train", streaming=True)
        names = det_category_names(ds.features)
        for i, row in enumerate(undecoded(ds)):
            if n >= YOLO_CAP_PER_SOURCE:
                break
            objs = row["objects"]
            boxes = []
            for bb, c in zip(objs["bbox"], objs["categories"]):
                k = det_class(spec, names[c] if names else c)
                if k is not None:
                    boxes.append((k, *map(float, bb)))
            if boxes:
                n += write_det(open_hf_image(row["image"]), boxes, pick_split(f"{name}{i}"), f"{name}__{n:05d}")
    det_done(name, n)


def ingest_plantdoc_det():
    name, n = "det_plantdoc", 0
    clone_plantdoc()
    pd_dir = PLANTDOC_DIR
    for split_dir, csv_name, official in (("TRAIN", "train_labels.csv", None), ("TEST", "test_labels.csv", "test")):
        frame = pd.read_csv(pd_dir / csv_name)
        lookup = {p.name: p for p in images_in(pd_dir / split_dir)}
        for k, (fname, g) in enumerate(frame.groupby("filename")):
            if DRY_RUN and k >= YOLO_CAP_PER_SOURCE:
                break
            src = lookup.get(fname)
            if src is None:
                continue
            try:
                img = Image.open(src).convert("RGB")
            except Exception:
                continue
            W, H = img.size
            boxes = []
            for _, r in g.iterrows():
                try:
                    x1, y1, x2, y2 = plantdoc_boxes(r, W, H)
                except (ValueError, ZeroDivisionError):
                    continue
                boxes.append((0, x1, y1, x2 - x1, y2 - y1))
            split = official or ("val" if zlib.crc32(fname.encode()) % 10 == 0 else "train")
            n += write_det(img, boxes, split, f"{name}__{n:05d}")
    det_done(name, n)


for spec in [dict(name="det_plantdoc")] + DET_SOURCES:
    if det_restore(spec["name"]):
        print(f"[{spec['name']}] restored")
        continue
    try:
        ingest_plantdoc_det() if spec["name"] == "det_plantdoc" else ingest_det(spec)
    except Exception as e:
        for p in YOLO_DIR.rglob(f"{spec['name']}__*"):
            p.unlink(missing_ok=True)
        set_status(spec["name"], "FAILED", note=repr(e))

DET_SOURCE_CLASSES = {"det_plantdoc": {0}}
for s in DET_SOURCES:
    DET_SOURCE_CLASSES[s["name"]] = {det_class(s, None)} if isinstance(s["map"], str) else set(s["map"].values())

DATA_YAML = YOLO_DIR / "data.yaml"
DATA_YAML.write_text("\n".join([f"path: {YOLO_DIR}", "train: images/train", "val: images/val", "test: images/test", "names:"] +
                               [f"  {k}: {v}" for k, v in DET_NAMES.items()]))
for s in ("train", "val", "test"):
    print(s, len(list((YOLO_DIR / "images" / s).glob("*.jpg"))), "images")
print()
status_table()
assert any((YOLO_DIR / "images" / "train").iterdir()) and any((YOLO_DIR / "images" / "val").iterdir()), "Detector has no train/val images"

In [ ]:
from ultralytics import YOLO

YOLO_RUNS = ROOT / "yolo_runs"


def train_yolo(name, weights, epochs, data):
    last = YOLO_RUNS / name / "weights" / "last.pt"
    best = YOLO_RUNS / name / "weights" / "best.pt"
    if best.exists() and (YOLO_RUNS / name / "done").exists():
        return best
    if last.exists():
        YOLO(str(last)).train(resume=True)
    else:
        YOLO(weights).train(data=str(data), epochs=epochs, imgsz=640, batch=32, patience=30, cos_lr=True, close_mosaic=10,
                            mosaic=1.0, mixup=0.1, seed=SEED, project=str(YOLO_RUNS), name=name, exist_ok=True)
    (YOLO_RUNS / name / "done").write_text("ok")
    return best


best_r1 = train_yolo("round1", "yolo11n.pt", YOLO_EPOCHS, DATA_YAML)

# Pseudo-labels: add confident boxes for classes the image's source does not annotate
PSEUDO = Path(f"/content/yolo_v2_pseudo{SUFFIX}")
if not (PSEUDO / "data.yaml").exists():
    shutil.rmtree(PSEUDO, ignore_errors=True)
    shutil.copytree(YOLO_DIR, PSEUDO)
    det = YOLO(str(best_r1))
    added = Counter()
    for split in ("train", "val"):
        imgs = sorted((PSEUDO / "images" / split).glob("*.jpg"))
        for k in range(0, len(imgs), 64):
            batch = imgs[k:k + 64]
            for p, res in zip(batch, det.predict([str(x) for x in batch], conf=0.6, verbose=False)):
                src = p.stem.split("__")[0]
                lab = PSEUDO / "labels" / split / f"{p.stem}.txt"
                existing = [list(map(float, l.split()[1:5])) for l in lab.read_text().splitlines()] if lab.exists() else []
                new = []
                for c, xywhn in zip(res.boxes.cls.tolist(), res.boxes.xywhn.tolist()):
                    if int(c) in DET_SOURCE_CLASSES.get(src, set()):
                        continue
                    cx, cy, w, h = xywhn
                    overlap = any(abs(cx - e[0]) < (w + e[2]) / 2 and abs(cy - e[1]) < (h + e[3]) / 2 for e in existing)
                    if not overlap:
                        new.append(f"{int(c)} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
                        added[DET_NAMES[int(c)]] += 1
                if new:
                    with open(lab, "a") as f:
                        f.write(("\n" if existing else "") + "\n".join(new))
    (PSEUDO / "data.yaml").write_text(DATA_YAML.read_text().replace(str(YOLO_DIR), str(PSEUDO)))
    print("Pseudo-labels added:", dict(added))

best_r2 = train_yolo("round2", str(best_r1), YOLO_PSEUDO_EPOCHS, PSEUDO / "data.yaml")

# Per-class mAP on test images from sources that annotate that class
DET_REPORT = {}
final_det = YOLO(str(best_r2))
test_imgs = sorted((YOLO_DIR / "images" / "test").glob("*.jpg"))
for c, cname in DET_NAMES.items():
    imgs = [str(p) for p in test_imgs if c in DET_SOURCE_CLASSES.get(p.stem.split("__")[0], set())]
    if not imgs:
        continue
    lst = YOLO_DIR / f"test_{cname}.txt"
    lst.write_text("\n".join(imgs))
    yml = YOLO_DIR / f"test_{cname}.yaml"
    yml.write_text(DATA_YAML.read_text().replace("test: images/test", f"test: {lst}"))
    try:
        m = final_det.val(data=str(yml), split="test", verbose=False)
        idx = list(m.box.ap_class_index)
        DET_REPORT[cname] = dict(images=len(imgs), map50=float(m.box.ap50[idx.index(c)]) if c in idx else None,
                                 map50_95=float(m.box.ap[idx.index(c)]) if c in idx else None)
    except Exception as e:
        print(f"WARNING detector test for '{cname}' failed: {e!r}")
        DET_REPORT[cname] = dict(images=len(imgs), map50=None, map50_95=None)
print(json.dumps(DET_REPORT, indent=1))

det_onnx = final_det.export(format="onnx", imgsz=640, opset=17, simplify=True, dynamic=False)
DET_ONNX = EXPORT / "yolo11n_leaf.onnx"
shutil.copy(det_onnx, DET_ONNX)
dsess = ort.InferenceSession(str(DET_ONNX), providers=["CPUExecutionProvider"])
out_shape = dsess.get_outputs()[0].shape
assert dsess.get_inputs()[0].name == "images" and list(dsess.get_inputs()[0].shape) == [1, 3, 640, 640]
assert out_shape[1] == 4 + len(DET_NAMES), out_shape
DETECTION_CLASSES = {str(k): v for k, v in final_det.names.items()}
assert DETECTION_CLASSES == {str(k): v for k, v in DET_NAMES.items()}
REPORT["detector"] = DET_REPORT
REPORT["detector_onnx_mb"] = DET_ONNX.stat().st_size / 1e6
REPORT["detector_cpu_ms_1thread"] = bench(DET_ONNX, (1, 3, 640, 640), 1)
print("Detector output", out_shape, "classes", DETECTION_CLASSES)

## 9. classes.json with treatments, report, and download

In [ ]:
CROP_NAMES = {"maize": "Corn (Maize)", "bitter_gourd": "Bitter Gourd", "bottle_gourd": "Bottle Gourd", "date_palm": "Date Palm",
              "citrus": "Citrus", "pepper": "Pepper"}
PART_NAMES = {"leaf": "leaf", "fruit": "fruit", "tuber": "tuber", "stem": "stem", "panicle": "panicle", "ear": "ear", "head": "head",
              "bulb": "bulb", "plant": "plant", "nut": "nut"}

SPECIFIC = {
    "rice_leaf___blast": ("High", "Apply tricyclazole or azoxystrobin at first lesions and at booting. Avoid excess nitrogen; use resistant varieties."),
    "rice_leaf___bacterial_leaf_blight": ("High", "No curative spray. Drain fields periodically, balance N with K, use resistant varieties and clean seed."),
    "rice_leaf___tungro": ("Critical", "Control green leafhopper vectors early, rogue infected hills, synchronise planting and use resistant varieties."),
    "rice_stem___sheath_blight": ("Moderate", "Reduce plant density and nitrogen; apply validamycin or hexaconazole when sheath lesions appear."),
    "wheat_leaf___yellow_rust": ("High", "Spray a triazole (tebuconazole or propiconazole) at first pustules; grow resistant cultivars."),
    "wheat_leaf___brown_rust": ("Moderate", "Apply triazole or strobilurin fungicide at flag-leaf stage if pustules spread; destroy volunteer wheat."),
    "wheat_leaf___septoria": ("Moderate", "Protect flag leaf with azole + SDHI fungicide; rotate crops and bury infected stubble."),
    "potato_leaf___late_blight": ("Critical", "URGENT: remove infected foliage, spray mandipropamid or fluopicolide every 5-7 days in wet weather."),
    "tomato_leaf___late_blight": ("Critical", "URGENT: remove infected plants, spray mandipropamid or chlorothalonil, keep foliage dry."),
    "cassava_leaf___mosaic_disease": ("Critical", "Use virus-free cuttings, rogue infected plants early and control whiteflies."),
    "banana_leaf___fusarium_wilt": ("Critical", "No chemical cure. Remove and destroy infected plants, quarantine the area, use resistant cultivars."),
    "citrus_leaf___greening": ("Critical", "No cure. Remove infected trees, control Asian citrus psyllid, plant certified disease-free stock."),
    "coffee_leaf___leaf_rust": ("High", "Spray copper or triazole fungicide before the rainy season; prune for airflow; plant resistant varieties."),
    "maize_leaf___fall_armyworm": ("High", "Scout whorls; apply emamectin benzoate or Bt at early larval stage; encourage natural enemies."),
    "maize_leaf___lethal_necrosis": ("Critical", "Rogue infected plants, control thrips and aphid vectors, use certified seed and crop rotation."),
    "maize_leaf___streak_virus": ("High", "Control leafhoppers, avoid late planting next to old maize, use tolerant hybrids."),
}

RULES = [
    ("sigatoka", "High", "Remove and burn spotted leaves, keep drainage and spacing good, and rotate mancozeb with a triazole or strobilurin."),
    ("esca", "High", "Prune out infected wood in dry weather, protect pruning wounds and remove dead vines."),
    ("die_back", "High", "Cut 10 cm below dead tissue, burn prunings and spray copper oxychloride on cuts."),
    ("gummosis", "High", "Scrape out the gummy lesion, paint with Bordeaux paste and avoid trunk wounds and waterlogging."),
    ("shot_hole", "Moderate", "Prune infected twigs, apply copper at leaf fall and bud swell, and avoid overhead irrigation."),
    ("scorch", "Moderate", "Remove infected leaves, avoid overhead watering and apply captan or a copper fungicide."),
    ("leaf_mold", "Moderate", "Lower greenhouse humidity, ventilate and space plants, and spray chlorothalonil or copper."),
    ("gray_mold", "Moderate", "Remove infected tissue, ventilate well and apply fenhexamid or a biological Bacillus product."),
    ("white_mold", "High", "Remove infected plants with the soil around them, widen spacing and rotate away from host crops."),
    ("purple_blotch", "Moderate", "Spray mancozeb or a strobilurin at first lesions, avoid overhead irrigation and rotate crops."),
    ("black_scurf", "Moderate", "Plant certified seed tubers, avoid cold wet soils at planting and harvest promptly."),
    ("leaf_scald", "Moderate", "Avoid excess nitrogen, use clean seed and spray a fungicide if lesions spread."),
    ("pokkah", "Moderate", "Remove affected shoots and spray carbendazim or copper oxychloride in early stages."),
    ("chlorosis", "Low", "Check nutrition and drainage; remove severely affected leaves."),
    ("yellow_leaf", "High", "Use virus-free planting material, control aphid vectors and rogue infected stools."),
    ("grassy_shoot", "High", "Remove infected clumps, use hot-water-treated setts and control leafhopper vectors."),
    ("viral", "Critical", "No cure. Rogue infected plants, control insect vectors and use certified clean planting material."),
    ("insect_damage", "Moderate", "Inspect the plants for the pest, remove damaged leaves and use neem or a targeted insecticide."),
    ("melanose", "Low", "Prune dead twigs where the fungus survives and apply copper after petal fall."),
    ("dot", "Moderate", "Remove infected fruit and leaves and spray copper oxychloride during fruit set."),
    ("necrosis", "Moderate", "Remove infected leaves, control insect vectors and use resistant varieties."),
    ("reddening", "Low", "Usually stress or nutrient deficiency (magnesium or nitrogen): test the soil and correct it."),
    ("variegation", "Low", "Remove affected plants if it spreads; check for virus vectors."),
    ("scars", "Low", "Usually mechanical or insect damage; protect plants and monitor for pests."),
    ("rust", "Moderate", "Apply a triazole or strobilurin fungicide at first pustules and remove heavily infected leaves."),
    ("late_blight", "Critical", "URGENT: remove infected tissue and spray an oomycete fungicide (mandipropamid, fluopicolide)."),
    ("early_blight", "Moderate", "Remove lower infected leaves, mulch, and spray chlorothalonil or mancozeb every 7-10 days."),
    ("blight", "Moderate", "Remove infected leaves, improve airflow and apply a protectant fungicide (copper or mancozeb)."),
    ("mildew", "Moderate", "Improve airflow; spray sulfur, potassium bicarbonate or a systemic fungicide at first signs."),
    ("virus", "Critical", "No cure. Rogue infected plants, control insect vectors and use certified clean planting material."),
    ("mosaic", "Critical", "No cure. Rogue infected plants, control aphid/whitefly vectors and disinfect tools."),
    ("greening", "Critical", "No cure. Remove infected trees and control psyllid vectors."),
    ("wilt", "High", "Remove infected plants, avoid waterlogging, rotate crops and use resistant varieties."),
    ("rot", "High", "Remove and destroy affected parts, avoid wounds and excess moisture, store produce cool and dry."),
    ("rotten", "High", "Discard rotten produce, sanitise storage and handle gently to avoid bruising."),
    ("damaged", "Low", "Sort out damaged produce; check for pests and handling or storage damage."),
    ("canker", "High", "Prune infected parts in dry weather, disinfect tools and apply copper sprays."),
    ("scab", "Moderate", "Apply captan or a DMI fungicide from bud break; clear fallen leaves."),
    ("anthracnose", "Moderate", "Remove infected parts, avoid overhead irrigation and spray copper or azoxystrobin."),
    ("bacterial", "Moderate", "Spray fixed copper, avoid overhead irrigation and working with wet plants, use clean seed."),
    ("spot", "Moderate", "Remove infected leaves, improve airflow and apply a protectant fungicide."),
    ("mite", "High", "Spray water or horticultural oil on leaf undersides, release predatory mites, use a miticide if severe."),
    ("thrips", "High", "Use blue sticky traps, spinosad or neem, and remove weeds that host thrips."),
    ("aphid", "Moderate", "Spray insecticidal soap or neem; encourage ladybirds and lacewings."),
    ("armyworm", "High", "Apply Bt or emamectin benzoate at early larval stage; scout regularly."),
    ("borer", "High", "Remove and destroy bored fruits/shoots, use pheromone traps and targeted insecticides."),
    ("miner", "Moderate", "Remove mined leaves, use yellow sticky traps and spinosad if needed."),
    ("hispa", "Moderate", "Clip infested leaf tips, and apply insecticide only above economic threshold."),
    ("beetle", "Moderate", "Hand-pick or use row covers; apply neem or pyrethrin if damage is heavy."),
    ("weevil", "Moderate", "Prune infested shoots and apply a targeted insecticide."),
    ("midge", "Moderate", "Prune galled shoots and apply a targeted insecticide at flushing."),
    ("bug", "Moderate", "Prune for airflow, monitor and apply a targeted insecticide if needed."),
    ("helopeltis", "Moderate", "Prune for airflow, monitor and apply a targeted insecticide if needed."),
    ("leaffolder", "Moderate", "Avoid excess nitrogen and apply insecticide only above economic threshold."),
    ("semilooper", "Moderate", "Apply Bt or a targeted insecticide when larvae are young."),
    ("grasshopper", "Moderate", "Till to destroy egg pods; use baits or insecticide on heavy infestations."),
    ("jassid", "Moderate", "Use resistant varieties, yellow sticky traps and neem or imidacloprid if needed."),
    ("mealybug", "Moderate", "Prune infested parts, spray neem or insecticidal soap, release Cryptolaemus beetles."),
    ("deficiency", "Low", "Run a soil or leaf test and correct the deficient nutrient with balanced fertiliser."),
    ("herbicide", "Low", "Avoid spray drift, rinse sprayers well and let the plant recover."),
    ("smut", "High", "Remove smutted whips before they open, use hot-water-treated setts and resistant varieties."),
    ("sooty", "Low", "Control the sap-sucking insects producing honeydew; wash leaves with mild soap."),
    ("sunscald", "Low", "Keep enough leaf cover over fruit and use shade nets in heat waves."),
    ("blossom_end_rot", "Moderate", "Water evenly and keep calcium available; avoid excess nitrogen."),
    ("cracking", "Low", "Water evenly to avoid sudden swelling of fruit."),
    ("bruising", "Low", "Handle tubers gently at harvest and in storage."),
    ("dead_heart", "High", "Control stem borers with pheromone traps and timely insecticide at egg hatch."),
]


def treatment_for(cls, cond):
    if cls in SPECIFIC:
        return SPECIFIC[cls]
    for key, sev, text in RULES:
        if key in cond:
            return sev, text
    return "Moderate", "Isolate affected plants, remove infected parts and consult a local agronomist for a targeted treatment."


classification_classes = []
for idx, cls in enumerate(CLASSES):
    if cls == "unknown":
        classification_classes.append(dict(id=idx, class_name=cls, species="Unknown", disease="Unknown", is_healthy=False, severity="None",
                                           treatment="Not a supported crop part. Retake the photo of a single leaf, fruit or ear in good light.",
                                           plant_part="none"))
        continue
    head, cond = cls.split("___")
    crop, part = head.rsplit("_", 1)
    healthy = cond == HEALTHY
    sev, text = ("None", "Healthy. Keep regular irrigation, nutrition and weekly scouting.") if healthy else treatment_for(cls, cond)
    classification_classes.append(dict(id=idx, class_name=cls, species=CROP_NAMES.get(crop, crop.replace("_", " ").title()),
                                       disease="None" if healthy else cond.replace("_", " ").title(), is_healthy=healthy,
                                       severity=sev, treatment=text, plant_part=PART_NAMES.get(part, part)))

with open(EXPORT / "classes.json", "w", encoding="utf-8") as f:
    json.dump({"detection_classes": DETECTION_CLASSES, "classification_classes": classification_classes}, f, indent=2)

crops = sorted({c["species"] for c in classification_classes if c["class_name"] != "unknown"})
parts = sorted({c["plant_part"] for c in classification_classes if c["class_name"] != "unknown"})
REPORT.update(crops=crops, parts=parts, dropped_small_classes=dropped_small.to_dict(), weak_classes=weak.index.tolist(),
              top_confusions=[f"{a} -> {b} ({n_})" for n_, a, b in top_confusions], source_status=STATUS,
              test_by_crop=crop_acc.to_dict(), threshold_table=thr_table.to_dict("records"))
with open(EXPORT / "report.json", "w") as f:
    json.dump(REPORT, f, indent=1, default=float)

lines = [
    "# AgriTech v2 model report", "",
    f"- Classes: {NC} ({len(crops)} crops, parts: {', '.join(parts)})",
    f"- Real-world (field) test top-1: {REPORT['test_field_top1']:.3f}",
]
for kind in ["cross_source", "in_source", "official", "negatives"]:
    r = REPORT.get(f"test_{kind}")
    if r:
        lines.append(f"- Test {kind}: n={r['n']}, top-1={r['top1']:.3f}, coverage at threshold={r['coverage_at_threshold']:.3f}, "
                     f"precision at threshold={r['precision_at_threshold'] if r['precision_at_threshold'] is None else round(r['precision_at_threshold'], 3)}")
lines += [
    f"- Lab validation top-1: {REPORT['val_lab_top1']}",
    f"- ECE before/after temperature scaling: {ECE_BEFORE:.3f} / {ECE_AFTER:.3f} (T={TEMPERATURE:.2f})",
    f"- Recommended CLASSIFIER_CONF_THRESHOLD: {THRESHOLD} (validation precision >= {TARGET_PRECISION})",
    f"- Classifier ONNX: {REPORT['classifier_onnx_mb']:.1f} MB, CPU {REPORT['classifier_cpu_ms_1thread']:.1f} ms (1 thread), {REPORT['classifier_cpu_ms_4threads']:.1f} ms (4 threads)",
    f"- Detector ONNX: {REPORT['detector_onnx_mb']:.1f} MB, CPU {REPORT['detector_cpu_ms_1thread']:.1f} ms (1 thread), classes {DETECTION_CLASSES}",
    f"- Detector per-class test: {json.dumps(DET_REPORT)}", "",
    "## Weak classes", *[f"- {c}" for c in REPORT["weak_classes"]], "",
    "## Top confusions", *[f"- {c}" for c in REPORT["top_confusions"]], "",
    "## Dropped (too few images)", *[f"- {k}: {v}" for k, v in REPORT["dropped_small_classes"].items()], "",
    "## Install",
    "- yolo11n_leaf.onnx, mobilenetv4_plant_disease.onnx, classes.json -> apps/ai-service/app/models/",
    "- mobilenetv4_plant_disease.onnx, classes.json -> apps/frontend/public/models/",
    f"- Set CLASSIFIER_CONF_THRESHOLD (config.py) and CONFIDENCE_THRESHOLD (edge-inference.ts) to {THRESHOLD}",
]
(EXPORT / "report.md").write_text("\n".join(lines))
for fig in FIGS.glob("*.png"):
    shutil.copy(fig, EXPORT / fig.name)
print((EXPORT / "report.md").read_text())

# Attribution for CC BY sources: one line per source plus per-image credits for Open Images
credits = df.groupby("source").agg(images=("path", "size"), license=("license", "first")).reset_index()
credits.to_csv(EXPORT / "DATASET_CREDITS.csv", index=False)
df[df.attribution != ""][["source", "attribution"]].to_csv(EXPORT / "IMAGE_ATTRIBUTIONS.csv", index=False)

print()
status_table()
if DRY_RUN:
    print("DRY RUN finished: the whole pipeline ran end to end. Fix any FAILED/EMPTY sources above, "
          "then set DRY_RUN = False and run all again. The numbers above are meaningless (tiny data, 1 epoch).")
else:
    zip_path = shutil.make_archive("/content/agritech_models_v2", "zip", EXPORT)
    from google.colab import files
    files.download(zip_path)